## Generation of the a parcel based building inventory for New Jersey
This notebook walks through the generation of a refined building inventory to be used for risk assessment for the full state of New Jersey. The basic premise is that data from the National Structure Inventory is insufficient and is poorly quality controlled. Thus, we can use other available information like building footprints and parcel data to dramatically improve it. 

While this is not as complete or accurate as some other inventories, it does not use any proprietary data (e.g. First Street, CoreLogic, the now dead ZTRAX data, Regrid), making this a good example of how the NSI can be refined for larger regions than a single census block or city, in places where *minimal but sufficient* alternative data is avaialable. The goal is to be balance improvements in quality with transparency in the production of the inventory, which for many papers using commercial datasets is not always evident.

Note that the data is designed to assess the STRUCTURE losses only, and does not account for content value. Note that in NSI (the only other source of this information) there is some defined mapping of structure values to estimated contents value. This could be implemented at a later date if it were deemed important by the user. See section 6.1 in https://www.fema.gov/sites/default/files/documents/fema_hazus-6-inventory-technical-manual.pdf for more.

The datasets used in this analysis are:
* The 2022 National Structure Inventory: U.S. Army Corps of Engineers (2022). National Structure Inventory (NSI) Base Data. U.S. Army Corps of Engineers. https://nsi.sec.usace.army.mil/
* Footprints from OpenBuildingMap: Oostwegel, L. J., Schorlemmer, D., & Guéguen, P. (2025). From Footprints to Functions: A Comprehensive Global and Semantic Building Footprint Dataset. Scientific Data, 12(1), 1699.
* NJ MOD-IV Tax Assessor and Merged Parcel Data: ArcGIS. (2026). Arcgis.com. https://pumagic.maps.arcgis.com/home/item.html?id=406cf6860390467d9f328ed19daa359d

The methods in this notebook draw heavily on the methods employed in the following publications:
* Pollack, A., Benedict, J., Deb, M., Doss-Gollin, J., Judi, D., Lehman, W., ... & Keller, K. (2025). Unrefined national building inventories can mislead risk assessments and decisions. Available at SSRN 5575271.
* Pollack, A., Doss-Gollin, J., Srikrishnan, V., & Keller, K. (2025). UNSAFE: An UNcertain Structure And Fragility Ensemble framework for property-level flood risk estimation. Journal of Open Source Software, 10(115), 7527.
* Kijewski-Correa, T., Cetiner, B., Zhong, K., Wang, C., Zsarnoczay, A., Guo, Y., ... & McKenna, F. (2023). Validation of an augmented parcel approach for hurricane regional loss assessments. Natural Hazards Review, 24(3), 04023022.
* Kijewski-Correa, T., Taflanidis, A., Vardeman, C., Sweet, J., Zhang, J., Snaiki, R., ... & Kennedy, A. (2020). Geospatial environments for hurricane risk assessment: applications to situational awareness and resilience planning in New Jersey. Frontiers in Built Environment, 6, 549106.
* Zsarnóczay, A., Deierlein, G. G., McKenna, F., Schoettler, M., Yi, S. R., Cetiner, B., ... & DeJong, M. (2025). An open-source simulation platform to support and foster research collaboration in natural hazards engineering. Frontiers in Built Environment, 11, 1590479.

In each section of this notebook, decisions will be justified and the process explained, but I recognize this is a bit dense due to the huge amounts of gross regex parsing I had to do. To address this, I will make a second notebook which links to a GIS map and includes graphics demonstrating key decisions and potential issues in the data.


In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pandas as pd
import geopandas as gpd
import re

### Part 1 - Reading and Cleaning Data
In this section we do pretty standard operations to read in and clean data.
The OBM footprints were already on the HPC, so they are processed in the identical way in this notebook, as you will see shortly.

In [2]:
DATA_PATH = "/scratch/gpfs/GVILLARI/lt0663/risk_analysis/nj_parcels"
NSI_PATH = f"{DATA_PATH}/nsi_new_jersey.gpkg"
NJ_PARCEL_PATH = f"{DATA_PATH}/parcels_MOD4_Statewide.gdb/Statewide_Parcels_MODIV.gdb"
NJ_SHP_PATH = f"{DATA_PATH}/NJ_SHP/NJ_State_Boundary.shp"
OBM_PATH = "/scratch/gpfs/GVILLARI/lt0663/risk_analysis/data/OpenBuildingMap/building.032010.gpkg"

In [3]:
nsi_nj = gpd.read_file(NSI_PATH)
parcels_nj = gpd.read_file(NJ_PARCEL_PATH, layer='Cad_parcel_mod4')

/home/lt0663/.conda/envs/wx_base/lib/python3.13/site-packages/pyogrio/raw.py:198: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


We want to keep and parse out any useful information from the parcel data, so that we can ensure NSI points match up with parcels in space and in type. Here we can inspect all the fields in the native data, most of which are not useful for our purposes.

In [4]:
nsi_nj.columns

Index(['fd_id', 'bid', 'cbfips', 'st_damcat', 'occtype', 'bldgtype',
       'num_story', 'sqft', 'found_type', 'found_ht', 'med_yr_blt',
       'val_struct', 'val_cont', 'val_vehic', 'ftprntid', 'ftprntsrc',
       'source', 'students', 'pop2amu65', 'pop2amo65', 'pop2pmu65',
       'pop2pmo65', 'o65disable', 'u65disable', 'x', 'y', 'firmzone',
       'grnd_elv_m', 'ground_elv', 'geometry'],
      dtype='object')

In [5]:
parcels_nj.columns

Index(['PAMS_PIN', 'PCL_MUN', 'PCLBLOCK', 'PCLLOT', 'PCLQCODE', 'PCLLASTUPD',
       'PIN_NODUP', 'GIS_PIN', 'CD_CODE', 'PROP_CLASS', 'COUNTY', 'MUN_NAME',
       'PROP_LOC', 'OWNER_NAME', 'ST_ADDRESS', 'CITY_STATE', 'ZIP_CODE',
       'LAND_VAL', 'IMPRVT_VAL', 'NET_VALUE', 'LAST_YR_TX', 'BLDG_DESC',
       'LAND_DESC', 'CALC_ACRE', 'ADD_LOTS1', 'ADD_LOTS2', 'FAC_NAME',
       'PROP_USE', 'BLDG_CLASS', 'DEED_BOOK', 'DEED_PAGE', 'DEED_DATE',
       'YR_CONSTR', 'SALES_CODE', 'SALE_PRICE', 'DWELL', 'COMM_DWELL',
       'OLD_PROPID', 'ZIP5', 'ZIP_PLUS4', 'PCL_PBDATE', 'PCL_GUID',
       'Shape_Length', 'Shape_Area', 'geometry'],
      dtype='object')

The first processing issue we need to deal with is the presence of "additional lots" in the parcel data. These are cases where an assessor wants to link a parcel to another as they share the same characteristics or are part of the same property. This is also the first place where we will see the non-standard use of supposedly standardized fields in the MOD-IV data, unfortunately creating uncertainty in our data. In this case, it seems many assessors have decided to use these `ADD_LOTS1` and `ADD_LOTS2` fields as random overflow for their other fields:

In [6]:
# merge any additional lots together by checking ADD_LOTS1 and ADD_LOTS2. This will either give the additional PCLLOTs for that PCL_MUN and PCLBLOCK (if a
# string of numbers like '2,3,4')
# or it might give another PAMS_PIN. If it is a string and not one of these we will just have to ignore as some assessors just use this field for overflow...
parcels_nj.ADD_LOTS1.value_counts()

ADD_LOTS1
2                      3355
4                      2288
6                      2128
8                      2075
10                     1962
                       ... 
38.12                     1
38.11                     1
812                       1
371,372,387.01,388        1
6,7,8,9,10,11,12,37       1
Name: count, Length: 55946, dtype: int64

While there is probably lots we can get from this which we are missing, it is extremely clunky to parse through all of this and these `ADD_LOTS` fields appear more random than other non-standard fields we will deal with later. As a result we will unfortunately have to accept that some data will be lost in this step. However, we will also gain a lot of data for correctly filled out, connected lots. 

In this case, we will accept lots as connected if their `ADD_LOTS` fields contain either a single number or string of numbers separated by commas, or if it lists a PAMS_PIN to connect to, although I think in the end this doesn't happen.

Then, we will go through and we will search for the parent and child lots and share their attributes across them evenly. While this will not catch all of them, and in the metadata we will show examples of this, it will help us with some large apartment complexes:

In [7]:
def parse_additional_lots(add_lots_str):
    """
    Parse ADD_LOTS field.
    Returns tuple: (type, values) where type is 'lots', 'pins', or None
    """
    if pd.isna(add_lots_str) or str(add_lots_str).strip() == '':
        return None, []
    
    s = str(add_lots_str).strip()
    
    # Pattern 1: Comma-separated lot numbers like '2,3,4' or '8,9,9.01' or just
    # a single number like '1'
    if re.match(r'^[\d.,\s]+$', s):
        lots = [x.strip() for x in s.split(',') if x.strip()]
        return 'lots', lots
    
    # Pattern 2: PAMS_PIN format - contains underscores
    if '_' in s:
        pins = [x.strip() for x in s.split(',') if x.strip()]
        if all('_' in p for p in pins):
            return 'pins', pins
    
    return None, []


def merge_additional_lots(parcels_gdf, debug_path=None):
    """
    Merge additional lots into parent parcels based on ADD_LOTS1/ADD_LOTS2.
    Only merges where valid matches are found. Leaves everything else unchanged.
    
    Parameters
    ----------
    parcels_gdf : GeoDataFrame
        Parcel data
    debug_path : str, optional
        If provided, save debug files to this directory
    """
    parcels = parcels_gdf.copy()
    pin_to_idx = dict(zip(parcels['PAMS_PIN'], parcels.index))
    
    # Track merges: child_idx -> parent_idx
    merges = {}
    
    # Stats
    stats = {'lots_parsed': 0, 'pins_parsed': 0, 'children_found': 0, 'children_missing': 0}

    # go over each of the parcels
    for idx, row in parcels.iterrows():
        parent_pin = row['PAMS_PIN']
        pcl_mun = row['PCL_MUN']
        pclblock = row['PCLBLOCK']
        
        for col in ['ADD_LOTS1', 'ADD_LOTS2']:
            # check if there are any additional lots in the data
            if col not in parcels.columns:
                continue
            
            lot_type, values = parse_additional_lots(row[col])
            
            if lot_type == 'lots':
                # go through and based on the lots we parsed out
                # construct what the PAMS_PIN (municipality_parcel_lot) should be
                stats['lots_parsed'] += len(values)
                for lot in values:
                    child_pin = f"{pcl_mun}_{pclblock}_{lot}"
                    if child_pin in pin_to_idx and child_pin != parent_pin:
                        merges[pin_to_idx[child_pin]] = idx
                        stats['children_found'] += 1
                    elif child_pin != parent_pin:
                        # if that lot doesn't actually exist we will have to drop
                        stats['children_missing'] += 1
            
            # repeat the same for PAMS_PINS
            elif lot_type == 'pins':
                stats['pins_parsed'] += len(values)
                for child_pin in values:
                    if child_pin in pin_to_idx and child_pin != parent_pin:
                        merges[pin_to_idx[child_pin]] = idx
                        stats['children_found'] += 1
                    elif child_pin != parent_pin:
                        stats['children_missing'] += 1
    
    # save debug files before merging (will be used in metaadata)
    if debug_path is not None:
        parents_with_merges = parcels.loc[list(set(merges.values()))].copy()
        children_merged = parcels.loc[list(merges.keys())].copy()
        
        # Add parent PIN to children for reference
        children_merged['parent_pin'] = children_merged.index.map(
            {child: parcels.loc[parent, 'PAMS_PIN'] for child, parent in merges.items()}
        )
        
        parents_with_merges.to_file(f"{debug_path}/debug_parent_parcels.gpkg")
        children_merged.to_file(f"{debug_path}/debug_child_parcels.gpkg")
        print(f"Saved debug files to {debug_path}")
    
    # merge geometries to make them into single unified parcel for later analysis
    for child_idx, parent_idx in merges.items():
        child_geom = parcels.loc[child_idx, 'geometry']
        parent_geom = parcels.loc[parent_idx, 'geometry']
        parcels.loc[parent_idx, 'geometry'] = parent_geom.union(child_geom)
    
    # get rid of the children bc now they are duplicates
    parcels_out = parcels.drop(index=list(merges.keys()))
    
    # Print stats summary so we can see what changed
    print("="*50)
    print("ADDITIONAL LOTS MERGE SUMMARY")
    print("="*50)
    print(f"Lot numbers parsed:      {stats['lots_parsed']}")
    print(f"PAMS_PINs parsed:        {stats['pins_parsed']}")
    print(f"Children found & merged: {stats['children_found']}")
    print(f"Children not in data:    {stats['children_missing']}")
    print(f"Parcels before:          {len(parcels_gdf)}")
    print(f"Parcels after:           {len(parcels_out)}")
    print(f"Parcels removed:         {len(parcels_gdf) - len(parcels_out)}")
    print("="*50)
    
    return parcels_out

In [8]:
%%time
parcels_nj = merge_additional_lots(parcels_nj, debug_path=None)

ADDITIONAL LOTS MERGE SUMMARY
Lot numbers parsed:      233982
PAMS_PINs parsed:        0
Children found & merged: 193548
Children not in data:    40157
Parcels before:          3475671
Parcels after:           3291889
Parcels removed:         183782
CPU times: user 2min 31s, sys: 4.64 s, total: 2min 36s
Wall time: 2min 36s


Now that we have made some effort to merge the connected parcels, we will clean up the dataframes a bit and drop data that is not needed or informative for manual checking. Since we are using the HAZUS DDFs, we will also clean up the occtypes to match HAZUS if they are in NACCS form (e.g. RES1-2SWB) as this stories and foundation data is encoded into the `num_story` and `found_type` columns in NSI.

In [9]:
cols_to_keep_parcels = ['PAMS_PIN', 'PROP_CLASS', 'COUNTY', 'PROP_LOC', 'GIS_PIN',
                        'LAND_VAL', 'IMPRVT_VAL', 'NET_VALUE', 'BLDG_DESC', 'LAND_DESC',
                        'PROP_USE', 'BLDG_CLASS', 'DWELL', 'COMM_DWELL', 'Shape_Length', 'Shape_Area',
                        'geometry']

cols_to_keep_nsi = ['fd_id', 'cbfips', 'st_damcat', 'occtype', 'bldgtype', 'num_story', 'found_type',
                    'found_ht', 'grnd_elv_m', 'ground_elv', 'geometry', 'val_struct', 'val_cont']

In [10]:
nsi_nj_clean = nsi_nj[cols_to_keep_nsi]
parcels_nj_clean = parcels_nj[cols_to_keep_parcels]

In [11]:
# we don't really need the occtype to include stories and basement because it is already in the data so we can replace with RES1 for simplicity
nsi_nj_clean["occtype"] = nsi_nj_clean["occtype"].str.replace(r"-.*$", "", regex=True)
nsi_nj_clean.head(5)

/home/lt0663/.conda/envs/wx_base/lib/python3.13/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


,fd_id,cbfips,st_damcat,occtype,bldgtype,num_story,found_type,found_ht,grnd_elv_m,ground_elv,geometry,val_struct,val_cont
0,548509648,340297280001017,RES,RES1,W,2.0,C,1.5,1.961297,6.434703,POINT (-74.08485 39.91152),356900.839,178450.419
1,548511068,340297131001010,RES,RES1,W,1.0,B,2.0,2.652716,8.703138,POINT (-74.10459 40.1006),293271.182,146635.591
2,548523256,340410311013002,RES,RES1,W,1.0,B,2.0,101.671425,333.567678,POINT (-75.03149 40.96023),423172.098,211586.049
3,548545953,340297310021035,RES,RES1,W,2.0,B,2.0,2.471919,8.109971,POINT (-74.14944 39.89421),384874.181,192437.090
4,548549026,340297140004002,RES,RES1,M,1.0,B,2.0,13.874615,45.520391,POINT (-74.15328 40.04653),285159.837,142579.918


There are some cases where structures might straddle a number of tax footprints, or where the centroid of the parcel is not representitive of the building. This is probably fairly inconsequential, but there are some cases such as large apartment complexes or industrial parks where this is probably important to account for. In theory we could use NSI to deal with this, but using footprints is a bit more rigorous. While in the CONUS product these footprints feed into the uncertainty, in this study we just use them for position in the parcel. 

We will see a couple of examples where this is particularly useful (e.g. nursing homes, large apartment complexes, townhouses) when we look at data quality at the end of this notebook. For now, the main drawback to note is that this is not foolproof as we still need the parcel data to provide certain attributes like general occtype and most importantly the structure and land value. If this is NULL for whatever reason (e.g. merging did not catch it or the occtype is incorrect in the parcel, NSI or both) there will still be some errors, but this is likely still better than incorrectly or overvalued NSI points.

In keeping with our workflow, we will read them in and clean them here, but we will not actually start moving around and disaggregating buildings until all the processing of the parcel and NSI data has been completed.

In [12]:
%%time
# here we will clean the OBM data if needed
CLEAN_OBM = False

if CLEAN_OBM:
    obm_tile = gpd.read_file(OBM_PATH)

    nj_mask = gpd.read_file(NJ_SHP_PATH)

    # now crop the footprints to the state
    if obm_tile.crs != nj_mask.crs:
        obm_tile = obm_tile.to_crs(nj_mask.crs)
    
    obm_tile["geometry"] = obm_tile.geometry.make_valid()
    nj_mask["geometry"] = nj_mask.geometry.make_valid()

    print("Geometry is valid")

    obm_tile.sindex
    nj_mask.sindex

    print("clipping")

    bounds = nj_mask.total_bounds
    obm_filtered = obm_tile.cx[bounds[0]:bounds[2], bounds[1]:bounds[3]]
    
    nj_buildings = obm_filtered.clip(nj_mask)

    # write it to file so we don't have to do this next time
    nj_buildings.to_file("/scratch/gpfs/GVILLARI/lt0663/risk_analysis/nj_parcels/obm_nj.gpkg")

else:
    nj_buildings = gpd.read_file("/scratch/gpfs/GVILLARI/lt0663/risk_analysis/nj_parcels/obm_nj.gpkg")
    nj_buildings

CPU times: user 18.1 s, sys: 1.7 s, total: 19.8 s
Wall time: 20 s


## Part 2 - Monte Carlo Simulation for First Floor Elevation (foundation height)
Eventually, we will use some information from NSI that the parcel data does not provide in most places. Specifically, the parcel data does not provide foundation heights and rarely provides information on the basements. Indeed, it is not standard for the parcels to contain any information about basements, but as we will see later, some assessors do provide this information in their non-standard descriptions. Following the UNSAFE framework, we will resample the foundation heights and types in NSI for use later. While we could resample foundation type from the tracts as is done in the Philly study, for simplicity and because we do not have detailed parcel or tract level distributions that are independent of NSI to draw from like they did, we will not.

For foundation height we will use a triangular distribution obtained by a FATHOM survey with the USACE tool. While the lack of transparency in the FATHOM papers is less than desirable, Adam Pollack also uses this in the UNSAFE paper and it is the best that we have so we will also use it. Because we aren't treating the other characteristics as uncertain here (this is not inherently a risk study) we will just produce quantiles for FFE and then in the final data we can choose which to take based on how conservative we want to be. Note that the "minimal" dataframe we ultimately produce will default to the median (q50).

In [13]:
len(nsi_nj_clean[nsi_nj_clean.found_type == 'C'])

277330

In [14]:
## first we define the triangular distribution parameters
## for each of the foundation types, taken from the UNSAFE
## code, which in turn is taken from FATHOM.
FFE_PARAMS = {
    'B': (0.0, 0.5, 1.5),   # Basement
    'C': (0.0, 1.5, 4),     # Crawl space
    'S': (0.0, 1.5, 4),     # Slab
    'I': (6.0, 9.0, 12.0),  # Pile
    'W': (6.0, 9.0, 12.0),  # Solid Wall
    'P': (6.0, 9.0, 12.0)   # pier
}

FND_TYPES = np.array(['B', 'C', 'S', 'I', 'W', 'P'])


In [15]:
def resample_ffe(df, found_type_col='found_type', n_sims=2000, seed=None):
    """
    Resample FFE based on foundation type using triangular distribution.
    Returns summary statistics (quantiles).
    
    Parameters
    ----------
    df : DataFrame
        Must have foundation type column with values 'B', 'C', or 'S'
    found_type_col : str
        Column name for foundation type
    n_sims : int
        Number of Monte Carlo simulations
    seed : int
        Random seed for reproducibility
    
    Returns
    -------
    DataFrame
        Original df with added columns: ffe_q05, ffe_q25, ffe_q50, ffe_q75, ffe_q95
    """
    rng = np.random.default_rng(seed)
    df = df.copy()
    n = len(df)
    
    ffe_sims = np.empty((n, n_sims))
    
    for i in range(n_sims):
        ffes = np.zeros(n)
        for fnd_type in ['B', 'C', 'S', 'I', 'W', 'P']:
            mask = df[found_type_col] == fnd_type
            if mask.any():
                left, mode, right = FFE_PARAMS[fnd_type]
                ffes[mask] = rng.triangular(left, mode, right, size=mask.sum())
        ffe_sims[:, i] = ffes
    
    df['ffe_q05'] = np.round(np.percentile(ffe_sims, 5, axis=1), 2)
    df['ffe_q25'] = np.round(np.percentile(ffe_sims, 25, axis=1), 2)
    df['ffe_q50'] = np.round(np.percentile(ffe_sims, 50, axis=1), 2)
    df['ffe_q75'] = np.round(np.percentile(ffe_sims, 75, axis=1), 2)
    df['ffe_q95'] = np.round(np.percentile(ffe_sims, 95, axis=1), 2)
    
    return df


In [16]:
%%time
nsi_nj_clean = resample_ffe(nsi_nj_clean)

CPU times: user 31min 33s, sys: 1min 29s, total: 33min 2s
Wall time: 33min 13s


In [17]:
nsi_nj_clean.found_type.unique()

array(['C', 'B', 'S', 'I', 'W', 'P'], dtype=object)

In [18]:
nsi_nj_clean.head(5)

,fd_id,cbfips,st_damcat,occtype,bldgtype,num_story,found_type,found_ht,grnd_elv_m,ground_elv,geometry,val_struct,val_cont,ffe_q05,ffe_q25,ffe_q50,ffe_q75,ffe_q95
0,548509648,340297280001017,RES,RES1,W,2.0,C,1.5,1.961297,6.434703,POINT (-74.08485 39.91152),356900.839,178450.419,0.53,1.22,1.75,2.43,3.33
1,548511068,340297131001010,RES,RES1,W,1.0,B,2.0,2.652716,8.703138,POINT (-74.10459 40.1006),293271.182,146635.591,0.22,0.44,0.64,0.88,1.22
2,548523256,340410311013002,RES,RES1,W,1.0,B,2.0,101.671425,333.567678,POINT (-75.03149 40.96023),423172.098,211586.049,0.21,0.44,0.66,0.89,1.22
3,548545953,340297310021035,RES,RES1,W,2.0,B,2.0,2.471919,8.109971,POINT (-74.14944 39.89421),384874.181,192437.090,0.19,0.43,0.63,0.88,1.22
4,548549026,340297140004002,RES,RES1,M,1.0,B,2.0,13.874615,45.520391,POINT (-74.15328 40.04653),285159.837,142579.918,0.19,0.42,0.63,0.88,1.24


Now, we can move on to the parcel data:

In [19]:
parcels_nj_clean.head(5)

,PAMS_PIN,PROP_CLASS,COUNTY,PROP_LOC,GIS_PIN,LAND_VAL,IMPRVT_VAL,NET_VALUE,BLDG_DESC,LAND_DESC,PROP_USE,BLDG_CLASS,DWELL,COMM_DWELL,Shape_Length,Shape_Area,geometry
0,0602_245_18.01,1,CUMBERLAND,2519 CHURCH STREET,0602_245_18.01,4500.0,0.0,4500.0,None,82.5 X 173,None,None,NaN,NaN,506.948414,14105.352192,"MULTIPOLYGON (((339378.925 150340.916, 339368...."
1,0602_256_4.02,4A,CUMBERLAND,1725 MAIN STREET,0602_256_4.02,20000.0,31800.0,51800.0,1SFCLA1UG,45X168.62 IRR,210,None,NaN,NaN,454.854546,7581.960830,"MULTIPOLYGON (((341022.808 150530.635, 340988...."
2,0602_256_4.01,15F,CUMBERLAND,1723 MAIN STREET,0602_256_4.01,18000.0,66400.0,84400.0,2SFCLA,35X174.87 IRR,770,None,NaN,NaN,445.761001,6566.822274,"MULTIPOLYGON (((340978.236 150536.824, 340952...."
3,0602_70_277,2,CUMBERLAND,247 MIST ROAD,0602_70_277,31600.0,49900.0,81500.0,1SFVLM,80 X 100,000,5214,1.0,NaN,362.125209,8098.973558,"MULTIPOLYGON (((337582.678 182335.442, 337559...."
4,0602_139_7902,15C,CUMBERLAND,BEAVER DR,0602_139_7902,3000.0,0.0,3000.0,None,40X100,None,None,NaN,NaN,271.939615,3771.512282,"MULTIPOLYGON (((337982.218 181371.2, 337889.23..."


## Part 3 - Parse the parcels! 
Now, we will parse what we can out of the parcel data. In this first part we will use the NHERI rule sets to assign an 
occtype that corresponds to what we would find in the NSI or in the HAZUS DDFs.

In [20]:
### look-up table for property types
def get_base_occupancy(prop_class):
    """
    map PROP_CLASS to base HAZUS occupancy type.
    This is our starting point before refining with BLDG_DESC/DWELL.

    This is default values based on the NHERI assessment for 
    Atlantic County, with some modifications based on my own judgement
    and the NJ Tax Apraisal Manual: 
    https://www.nj.gov/treasury/taxation/pdf/lpt/realpropertyappraisal.pdf

    Parameters
    ----------
    prop_class : string
        the property class for each parcel parsed from the parcels gdf

    Returns
    -------
        : str
        the occupancy type parsed from the property class
    """
    pc = str(prop_class).strip()
    
    mapping = {
        '1': None,        # Vacant land - no building
        '2': 'RES1',      # Residential - default to single family, refine later
        '3A': 'AGR1',     # Farm (regular)
        '3B': 'AGR1',     # Farm (qualified)
        '4A': 'COM1',     # Commercial
        '4B': 'IND1',     # Industrial
        '4C': 'RES3D',    # Apartment - refined with DWELL
        '5A': 'IND1',     # Railroad Class I
        '5B': 'IND1',     # Railroad Class II
        '6A': 'COM4',     # Telephone utility
        '6B': 'IND3',     # Petroleum refinery
        '15A': 'EDU1',    # Public school
        '15B': 'EDU1',    # Other school
        '15C': 'GOV1',    # Public property
        '15D': 'REL1',    # Church/charitable
        '15E': 'REL1',    # Cemetery
        '15F': 'GOV1',    # Other exempt
    }
    
    return mapping.get(pc, None)


In [21]:
def parse_dwell(dwell):
    """
    Convert DWELL to integer. Returns None if missing/invalid.

    I actually have some concerns that this is a counter-productive
    step since it seems a lot of parcels get misclassified and
    assigned DWELL == 1.0000, but we will keep for now since
    I know there are some apartment complexes where this is 
    helpful. It also seems like there are cases where the
    number of apartments could be inferred from the BLDG_DESC
    or from the random overflow that gets unhelpfully thrown
    into the ADD_LOTS1 and ADD_LOTS2 columns but there is already
    too much complicated regex already...

    Also maybe this is just not necessary at all... It got pared
    down many times from its original use to spit an updated occtype
    out and now it just reads and returns? Should see if the parsing
    breaks by removing...

    Parameters
    ----------
    dwell : float
        number of dwellings in the data parsed from the gdf

    Returns
    -------
        : float
        number of dwellings
    """
    if pd.isna(dwell):
        return None
    try:
        val = int(float(dwell))
        return val if val > 0 else None
    except:
        return None

In [22]:
def get_occupancy_from_prop_use(prop_use):
    """
    Map PROP_USE code to HAZUS occupancy type.
    Returns None if no mapping found (fall back to PROP_CLASS).
    
    Based on FEMA/NHERI Occupancy Class Rulesets for MOD IV Data.

    NOTE: can definitley come back to this and add more corresponding prop_uses
    as there are many more specific ones, which we can probably then use to get
    a more specific DDFs...

    Parameters
    ----------
    prop_use : string
        the property use code drawn from the gdf

    Returns
    -------
        : string
        HAZUS occtype pulled from the parcel data
    """
    PROP_USE_DIRECT = {
        '999': 'RES1',
        '512': 'RES2',
        '020': 'RES3B',
        '029': 'RES3C',
        '021': 'RES3E',
        '635': 'RES5',
        '335': 'RES5',
        '180': 'COM2',
        '562': 'COM7',
        '650': 'IND4',
        '191': 'GOV2',
        '075': 'EDU2',
        '074': 'REL1',
        '130': 'REL1',
    }
    
    # Temporary Lodging (RES4)
    PROP_USE_RES4 = {'280', '281', '282', '283', '530'}
    
    # Nursing Home (RES6)
    PROP_USE_RES6 = {'270', '273', '278', '636', '637'}
    
    # Retail Trade (COM1)
    PROP_USE_COM1 = {'525', '526', '527', '528', '529'}  # 525-529 per PDF
    
    # Personal and Repair Services (COM3)
    PROP_USE_COM3 = {'110', '210', '219', '220'}  # plus range 638-650
    
    # Business/Professional (COM4)
    PROP_USE_COM4 = {'190', '441', '760', '761', '500', '561', '563', '565', '566', '569'}
    
    # Banks (COM5)
    PROP_USE_COM5 = {'050', '051', '059'}
    
    # Hospitals (COM6)
    PROP_USE_COM6 = {'271', '272', '279'}
    
    # Parking (COM10)
    PROP_USE_COM10 = {'211', '212', '750'}
    
    # Food/Drugs/Chemicals (IND3)
    PROP_USE_IND3 = {'970', '940', '218', '571'}
    
    # Construction (IND6)
    PROP_USE_IND6 = {'040', '440', '580'}
    
    # Agriculture (AGR1)
    PROP_USE_AGR1 = {'120', '222', '430', '740'}
    
    # General Services (GOV1)
    PROP_USE_GOV1 = {'221', '564', '570', '230'}
    
    # Schools/Libraries (EDU1)
    PROP_USE_EDU1 = {'442', '660', '661'}
    
    if pd.isna(prop_use) or str(prop_use).strip() in ['', 'None']:
        return None
    
    code = str(prop_use).strip().zfill(3)  # Pad to 3 digits
    
    # Check direct mappings first
    if code in PROP_USE_DIRECT:
        return PROP_USE_DIRECT[code]
    
    # Check set-based mappings
    if code in PROP_USE_RES4: return 'RES4'
    if code in PROP_USE_RES6: return 'RES6'
    if code in PROP_USE_COM1: return 'COM1'
    if code in PROP_USE_COM3: return 'COM3'
    if code in PROP_USE_COM4: return 'COM4'
    if code in PROP_USE_COM5: return 'COM5'
    if code in PROP_USE_COM6: return 'COM6'
    if code in PROP_USE_COM10: return 'COM10'
    if code in PROP_USE_IND3: return 'IND3'
    if code in PROP_USE_IND6: return 'IND6'
    if code in PROP_USE_AGR1: return 'AGR1'
    if code in PROP_USE_GOV1: return 'GOV1'
    if code in PROP_USE_EDU1: return 'EDU1'
    
    # Check range-based mappings
    try:
        code_int = int(code)
        
        # COM1: 525-529 plus others
        if 525 < code_int < 529: return 'COM1'
        
        # COM3: 638-650
        if 638 < code_int < 650: return 'COM3'
        
        # COM4: 729-740
        if 729 < code_int < 740: return 'COM4'
        
        # COM8: 609-630, 070-080, 510-540 (excluding specific codes)
        if 609 < code_int < 630: return 'COM8'
        if 70 < code_int < 73: return 'COM8'
        if 80 < code_int < 81: return 'COM8'
        if 510 < code_int < 511: return 'COM8'
        if 540 < code_int < 541: return 'COM8'
        if 769 < code_int < 773: return 'COM8'
        
        # COM9: 768-779
        if 768 < code_int < 779: return 'COM9'
        
        # IND1: 330 OR (939-961)
        if code_int == 330: return 'IND1'
        if 939 < code_int < 961: return 'IND1'
        
        # IND2: (010-031, 790-791) OR (949-961)
        if 10 < code_int < 31: return 'IND2'
        if 790 < code_int < 791: return 'IND2'
        if 949 < code_int < 961: return 'IND2'
        
    except ValueError:
        pass
    
    return None

In [23]:
def get_occupancy_from_bldg_class(bldg_class):
    """
    Map BLDG_CLASS to HAZUS occupancy type.
    Returns None if no mapping found.
    
    BLDG_CLASS is a numeric code describing building quality/type.

    See https://www.nj.gov/treasury/taxation/pdf/lpt/realpropertyappraisal.pdf
    page 50 for the R numbers alongside the NHERI rules.

    Parameters
    ----------
    bldg_class : string
        the building class taken from the gdf

    Returns
    -------
        : string
        HAZUS occtype pulled from the parcel data
    """
    if pd.isna(bldg_class) or str(bldg_class).strip() in ['', 'None']:
        return None
    
    try:
        bc = int(float(bldg_class))
    except (ValueError, TypeError):
        return None
    
    # Single Family Dwelling: BldgClass 11-32 (note reclassification from NHERI so that
    # townhouses are classified as multi-unit which I think is correct based on the DDFs
    # in HAZUS, not RES1 like in NHERI, because they share walls etc). Since this is 
    # inspired by the Philly study from Pollack et al 2025, and they define the twin 
    # row houses with neighbours as RES3, we will do the same. The legal definition of all 
    # of this is so messy and I think this makes more sense with the footprint, which appears
    # as a single building. (see https://github.com/abpoll/nsi_fit/blob/main/notebooks/explore.ipynb
    # which states that some buildings are coded as twin row homes, which we consider RES3)...
    if 11 < bc < 32:
        return 'RES1'
    
    # Row/Town houses (probably more thorough to do a check for how 
    # many parcels the actual footprint we are using intersects but
    # for now its probably better to just reclassify as RES3C (multi-family w 5-9)...
    if 32 < bc < 40:
        return 'RES3C'
        
    # Mobile Home: BldgClass 49-55
    if 49 < bc < 55:
        return 'RES2'
    
    # Multi-family 3-4 Units: BldgClass 42-50 OR BldgClass=145
    if (42 < bc < 50) or bc == 145:
        return 'RES3B'
    
    # Agriculture: BldgClass 151-165
    if 151 < bc < 165:
        return 'AGR1'
    
    return None


In [24]:
def refine_occupancy_from_units(current_occ, num_units, prop_class=None, bldg_class=None):
    """
    Refine residential occupancy type based on number of dwelling units.
    Only applies to residential types (RES1, RES3x).
    
    Doesn't allow:
    - 4C (apartment property class) to be overwritten (or they usually become RES1 by accident due to DWELL column)
    - Townhouses (bldg_class 33-39) to be overwritten (for tax purposes they are single family so get parsed as RES1
      but we want them to be RES3, with the RES3 class determined either by DWELL in the gdf or later the number of 
      parcels the associated foort
    
    Returns refined occupancy or original if not applicable.

    Parameters
    ----------
    current_occ : string
        the current occupancy type parsed from the gdf
    num_units : float or string
        the number of units parsed from the gdf DWELL column
    prop_class : string
        the property type, used to prevent the override of the 4C apartment class
    bldg_class : string
        the building class, used to prevent the override of RES3 townhomes we fixed
        in the get_occupancy_from_bldg_class

    Returns
    -------
    current_occ : string
        the updated occtype string
    """
    RESIDENTIAL_TYPES = {'RES1', 'RES3A', 'RES3B', 'RES3C', 'RES3D', 'RES3E', 'RES3F'}
    
    # Don't override apartments
    if str(prop_class).strip() == '4C':
        return current_occ
    
    # Don't override townhouses (bldg_class 33-39)
    if not (pd.isna(bldg_class) or str(bldg_class).strip() in ['', 'None']):
        try:
            bc = int(float(bldg_class))
            if 32 < bc < 40:
                return 'RES3C'
        except (ValueError, TypeError):
            pass  # Continue to next checks
    
    # Only refine residential types
    if current_occ not in RESIDENTIAL_TYPES:
        return current_occ
    
    # Parse units
    if pd.isna(num_units):
        return current_occ
    
    try:
        units = int(float(num_units))
    except (ValueError, TypeError):
        return current_occ
    
    # Apply unit-based classification
    if units == 1:
        return 'RES1'
    elif units == 2:
        return 'RES3A'
    elif units in [3, 4]:
        return 'RES3B'
    elif 5 <= units <= 9:
        return 'RES3C'
    elif 10 <= units <= 19:
        return 'RES3D'
    elif 20 <= units <= 49:
        return 'RES3E'
    elif units >= 50:
        return 'RES3F'
    
    return current_occ

In [25]:
def check_mobile_home(bldg_desc):
    """
    Check if BLDG_DESC indicates a mobile home.
    Returns True/False. Note that unfortunately, the mobile
    homes are largely misattributed as COM4 lots (i.e. prop_class 4A)
    in parcels and as RES1 in NSI. 
    
    This is probably because most 
    manufactured homes are in corporate owned and mixed-use parks where
    the leasing company owns the land and the residents own the home.
    This means that correctly identfiying mobile homes is extremely 
    diffcult and it is rare, even with the footprints, that our simple
    rules allow us to match them to parcels. This is explained more in the
    data quality section at the end of the notebook.

    There is definitely a Stephen Strader study on this focused on 
    tornados that we could use to make some progress on this, and there
    is probably a very important paper for us in this due to the high social
    vulnerability attributed to manufactured homes. For now though, we will
    just have to accept that we massively undercount RES2.
    
    Mobile home indicators:
    - Explicit words: MOBILE, TRAILER, MFD HOME
    - Style code M in coded format: 1SFM, 1SM, 1SALM, 1S-AL-M, etc.

    Parameters
    ----------
    bldg_desc : string
        the building description code parsed from the gdf

    Returns
    --------
        : bool
        either is a mobile home or not
    """
    if pd.isna(bldg_desc) or str(bldg_desc).strip() in ['', 'None']:
        return False
    
    desc = str(bldg_desc).upper().strip()
    
    # Explicit mobile home words
    if re.search(r'\bMOBILE\b|\bTRAILER\b|\bMFD\s*HOME\b|\bMANUFACTURED\b', desc):
        return True
    
    # Coded format: M as style code after stories+material
    # Patterns: 1SFM, 1SM, 1SALM, 1SBM, 1SCBM, 1S-F-M, 1S-AL-M, 1S-M
    
    # Pattern: [digits]S[material]M followed by end, space, digit, or garage indicator
    # This catches: 1SFM, 1SM, 1SALM, 1SBM, 1SALMG, 1SFM2G, etc.
    if re.match(r'^\d+\.?\d*S[FABCW]?[AL]?M(\s|$|\d|G|P|C)', desc):
        return True
    
    # Pattern with dashes: 1S-F-M, 1S-AL-M, 1S-M
    if re.match(r'^\d+\.?\d*S-[A-Z]*-?M(\s|$|-|\d)', desc):
        return True
    
    # Pattern: 1SM, 1SMG, 1SM-O
    if re.match(r'^\d+\.?\d*SM(\s|$|-|\d|G|P|C)', desc):
        return True
    
    return False

In [26]:
def get_hazus_occupancy(row):
    """
    Get HAZUS occupancy using all available data. This basically 
    calls all the functions listed above, so check those docstrings.

    Parameters
    ----------
    row : gdf.row
        row of the MOD-IV data we are iterating over

    Returns
    -------
    occ : string
        the final HAZUS occtype we parsed from these routines
    """
    
    # Step 1: Base from PROP_CLASS
    prop_class = row.get('PROP_CLASS')
    bldg_class = row.get('BLDG_CLASS')  
    
    occ = get_base_occupancy(prop_class)
    
    # Step 2: Override from PROP_USE
    occ_from_use = get_occupancy_from_prop_use(row.get('PROP_USE'))
    if occ_from_use:
        occ = occ_from_use
    
    # Step 3: Override from BLDG_CLASS
    occ_from_bldg = get_occupancy_from_bldg_class(bldg_class)
    if occ_from_bldg:
        occ = occ_from_bldg
    
    # Step 4: Check BLDG_DESC for mobile home -> RES2
    if check_mobile_home(row.get('BLDG_DESC')):
        return 'RES2'
    
    # Step 5: Refine by unit count (DWELL)
    occ = refine_occupancy_from_units(occ, row.get('DWELL'), prop_class, bldg_class)
    
    return occ

In [27]:
# apply it to the full data
parcels_nj_clean['PARSED_OCCTYPE'] = parcels_nj_clean.apply(get_hazus_occupancy, axis=1)

/home/lt0663/.conda/envs/wx_base/lib/python3.13/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [28]:
parcels_nj_clean.head(5)  # lets take a look at what we have now

,PAMS_PIN,PROP_CLASS,COUNTY,PROP_LOC,GIS_PIN,LAND_VAL,IMPRVT_VAL,NET_VALUE,BLDG_DESC,LAND_DESC,PROP_USE,BLDG_CLASS,DWELL,COMM_DWELL,Shape_Length,Shape_Area,geometry,PARSED_OCCTYPE
0,0602_245_18.01,1,CUMBERLAND,2519 CHURCH STREET,0602_245_18.01,4500.0,0.0,4500.0,None,82.5 X 173,None,None,NaN,NaN,506.948414,14105.352192,"MULTIPOLYGON (((339378.925 150340.916, 339368....",None
1,0602_256_4.02,4A,CUMBERLAND,1725 MAIN STREET,0602_256_4.02,20000.0,31800.0,51800.0,1SFCLA1UG,45X168.62 IRR,210,None,NaN,NaN,454.854546,7581.960830,"MULTIPOLYGON (((341022.808 150530.635, 340988....",COM3
2,0602_256_4.01,15F,CUMBERLAND,1723 MAIN STREET,0602_256_4.01,18000.0,66400.0,84400.0,2SFCLA,35X174.87 IRR,770,None,NaN,NaN,445.761001,6566.822274,"MULTIPOLYGON (((340978.236 150536.824, 340952....",COM8
3,0602_70_277,2,CUMBERLAND,247 MIST ROAD,0602_70_277,31600.0,49900.0,81500.0,1SFVLM,80 X 100,000,5214,1.0,NaN,362.125209,8098.973558,"MULTIPOLYGON (((337582.678 182335.442, 337559....",RES1
4,0602_139_7902,15C,CUMBERLAND,BEAVER DR,0602_139_7902,3000.0,0.0,3000.0,None,40X100,None,None,NaN,NaN,271.939615,3771.512282,"MULTIPOLYGON (((337982.218 181371.2, 337889.23...",GOV1


Now that we have replaced occtype by processing whatever we can from the parcel data, we will move on to other information we need for constructing HAZUS DDFs. Lots of this information is not supposed to actually be in the MOD-IV data, or is inference based on information from the architectural type courtesy of Lena (https://assets.website-files.com/5e405370054b238baa277415/62d9ed76332169f2cdb3a75d_Craftsman-Construction-Architectural-styles.pdf).

Note that we assume anything we pull out of this parcel data is going to override the NSI. In reality, this is inferrence for most cases, and we should probably be treating this as an uncertain characteristic to bootstrap. Nonetheless, our assumption for this study is that where the data is available, the parcel assessment is correct, so we will ultimately just use this to override the NSI where available. The characteristics we will search for here are:
* foundation type and thus presence of a basement (we are using average basement/no basement DDFs for simplcity)
* stories (the stories in NSI are uncertain so whatever information we can get will inform DDF selection)

In [29]:
def has_basement(bldg_desc):
    """
    Check if BLDG_DESC indicates a basement.
    Returns True/False/None (None = unknown).

    Does a regex search of the bldg_desc.
    Based on what I have seen we might want
    to add the ADD_LOTS fields too since some assessors
    seem to use these as overflow, but it will take time
    we don't have to inspect all these non-standard MOD-IV
    entries that don't match the official metadata...

    Parameters
    ----------
    bldg_desc : string
        the building description parsed in the gdf

    Returns
    -------
        : bool
        is a basement found or not?
    """
    if pd.isna(bldg_desc) or str(bldg_desc).strip() in ['', 'None']:
        return None
    
    desc = str(bldg_desc).upper().strip()
    
    # Basement indicators
    if re.search(r'\bBSMT\b|\bBASEMENT\b|\bBSM\b|\bBST\b|/B\b|W/B\b|\bFULL\s*BASE', desc):
        return True
    
    # No basement indicators
    if re.search(r'\bSLAB\b|\bSLB\b|\bCRAWL\b|\bNO\s*BSMT\b|\bNOBSMT\b', desc):
        return False
    
    return None

In [30]:
def parse_foundation(bldg_desc):
    """
    Extract foundation type from BLDG_DESC.
    Returns: 'B', 'S', 'C', or None.
    """
    if pd.isna(bldg_desc) or str(bldg_desc).strip() in ['', 'None']:
        return None
    
    desc = str(bldg_desc).upper().strip()
    
    # Check for basement
    if re.search(r'\bBSMT\b|\bBASEMENT\b|\bBSM\b|\bBST\b|/B\b|W/B\b|\bFULL\s*BASE', desc):
        return 'B'
    
    # Check for slab
    if re.search(r'\bSLAB\b|\bSLB\b|/SL\b', desc):
        return 'S'
    
    # Check for crawl space
    if re.search(r'\bCRAWL\b|\bCRWL\b', desc):
        return 'C'
    
    return None


In [31]:
def parse_stories(bldg_desc):
    """
    Extract number of stories from BLDG_DESC.

    This is the really big gross disgusting regex parsing
    function. Maybe there is a cleaner way but I basically
    just manually inspected the building descriptions in the
    MOD_IV data and looked at all the different ways the
    assessors entered information about the buildings. There
    is probably data loss/mis-classification here but manually
    checking the overall results suggests this is fairly 
    reasonable.

    We go in order of inference quality, returning the 
    standard MOD-IV entries first (i.e. pattersn that 
    actually follow the metadata), becoming more uncertain
    up to using buidling style (e.g. colonial, bungalow) to
    infer stories.

    Parameters
    ----------
    bldg_desc : string
        the building description parsed from the MOD-IV gdf

    Returns
    -------
        : string or None
        the number of stories regex parsing has pulled out of the data
    """
    if pd.isna(bldg_desc) or str(bldg_desc).strip() in ['', 'None', '0', '00', '000']:
        return None
    
    desc = str(bldg_desc).upper().strip()
    
    # Max reasonable stories - anything above this is likely sqft or other data
    MAX_STORIES = 500
    
    def validate_stories(val):
        """Return value if reasonable, else None to continue checking."""
        if val is not None and val <= MAX_STORIES:
            return val
        return None  # Continue to next pattern
    
    # Pattern: "1 1/2" at start (e.g., "1 1/2 S F", "1 1/2SF")
    match = re.match(r'^(\d+)\s*1/2', desc)
    if match:
        result = validate_stories(int(match.group(1)) + 0.5)
        if result is not None:
            return result
    
    # Pattern: "1.5S " or "2S " or "1.5S-" (with space or dash after S)
    match = re.match(r'^(\d+\.?\d*)S[\s\-/]', desc)
    if match:
        result = validate_stories(float(match.group(1)))
        if result is not None:
            return result
    
    # Pattern: "1SF", "2SF", "1.5SF", "2SFG" (S followed by F for frame)
    match = re.match(r'^(\d+\.?\d*)SF', desc)
    if match:
        result = validate_stories(float(match.group(1)))
        if result is not None:
            return result
    
    # Pattern: "1SB", "2SB" (S followed by B for brick)
    match = re.match(r'^(\d+\.?\d*)SB', desc)
    if match:
        result = validate_stories(float(match.group(1)))
        if result is not None:
            return result
    
    # Pattern: "1SCB", "2SCB" (S followed by CB for concrete block)
    match = re.match(r'^(\d+\.?\d*)SCB', desc)
    if match:
        result = validate_stories(float(match.group(1)))
        if result is not None:
            return result
    
    # Pattern: "1SS", "2SS" (S followed by S for stucco or structured steel)
    match = re.match(r'^(\d+\.?\d*)SS[^T]', desc)  # avoid matching SST (stone)
    if match:
        result = validate_stories(float(match.group(1)))
        if result is not None:
            return result
    
    # Pattern: "1SST", "2SST" (S followed by ST for stone)
    match = re.match(r'^(\d+\.?\d*)SST', desc)
    if match:
        result = validate_stories(float(match.group(1)))
        if result is not None:
            return result
    
    # Pattern: "1SAL", "2SAL" (S followed by AL for aluminum)
    match = re.match(r'^(\d+\.?\d*)SAL', desc)
    if match:
        result = validate_stories(float(match.group(1)))
        if result is not None:
            return result
    
    # Pattern: "1S-F-L", "2S-F-L" (with dashes between components)
    match = re.match(r'^(\d+\.?\d*)S-', desc)
    if match:
        result = validate_stories(float(match.group(1)))
        if result is not None:
            return result
    
    # Pattern: "2 STORY", "2-STORY", "1 STORY"
    match = re.search(r'(\d+\.?\d*)\s*-?\s*STOR', desc)
    if match:
        result = validate_stories(float(match.group(1)))
        if result is not None:
            return result
    
    # Pattern: "2STY", "1STY"
    match = re.search(r'(\d+\.?\d*)\s*STY', desc)
    if match:
        result = validate_stories(float(match.group(1)))
        if result is not None:
            return result
    
    # Pattern: "1 S F", "2 S B" (with spaces)
    match = re.match(r'^(\d+\.?\d*)\s+S\s+[FBACW]', desc)
    if match:
        result = validate_stories(float(match.group(1)))
        if result is not None:
            return result
    
    # Bi-level / Split = 2 stories
    if re.search(r'BI-?\s*LEVEL|BILEVEL|SPLIT\s*LEVEL|\bSPLIT\b|/S/L|S/L\b', desc):
        return -0.5 # note this is just a code for split level so i can use the split level DDF
    
    # Tri-level = 3 stories
    if re.search(r'TRI-?\s*LEVEL|TRILEVEL', desc):
        return 3.0
    
    # --- Fallback: infer from architectural style ---
    # Note: these are typical values, should be checked against NSI
    
    # Ranch/Rancher typically 1 story
    if re.search(r'\bRANCH\b|\bRANCHER\b', desc) and not re.search(r'RAISED\s*RANCH', desc):
        return 1.0
    
    # Cape Cod typically 1.5 stories
    if re.search(r'\bCAPE\b', desc):
        return 1.5
    
    # Colonial typically 2 stories
    if re.search(r'\bCOLONIAL\b', desc):
        return 2.0
    
    # Georgian typically 2 stories
    if re.search(r'\bGEORGIAN\b', desc):
        return 2.0
    
    # Bungalow typically 1 story
    if re.search(r'\bBUNGALOW\b|\bBUNGELOW\b', desc):
        return 1.0
    
    # Regency typically 2 stories
    if re.search(r'\bREGENCY\b', desc):
        return 2.0
    
    return None

In [32]:
def add_parsed_bldg_columns(gdf):
    """
    Function applies all the stuff in the functions above

    Parameters
    ----------
    gdf : gpd.geodataframe
        MOD-IV data

    Returns
    -------
    gdf : gpd.geodataframe
        MOD-IV dataframe with the inferred data
    """
    gdf = gdf.copy()
    
    gdf['PARSED_STORIES'] = gdf['BLDG_DESC'].apply(parse_stories)
    gdf['PARSED_HAS_BASEMENT'] = gdf['BLDG_DESC'].apply(has_basement)
    gdf['PARSED_FOUNDATION'] = gdf['BLDG_DESC'].apply(parse_foundation)
    
    return gdf

In [33]:
parcels_nj_parsed = add_parsed_bldg_columns(parcels_nj_clean) # apply the data

In [34]:
parcels_nj_parsed.head(5)  # now take a look at the data

,PAMS_PIN,PROP_CLASS,COUNTY,PROP_LOC,GIS_PIN,LAND_VAL,IMPRVT_VAL,NET_VALUE,BLDG_DESC,LAND_DESC,...,BLDG_CLASS,DWELL,COMM_DWELL,Shape_Length,Shape_Area,geometry,PARSED_OCCTYPE,PARSED_STORIES,PARSED_HAS_BASEMENT,PARSED_FOUNDATION
0,0602_245_18.01,1,CUMBERLAND,2519 CHURCH STREET,0602_245_18.01,4500.0,0.0,4500.0,None,82.5 X 173,...,None,NaN,NaN,506.948414,14105.352192,"MULTIPOLYGON (((339378.925 150340.916, 339368....",None,NaN,None,None
1,0602_256_4.02,4A,CUMBERLAND,1725 MAIN STREET,0602_256_4.02,20000.0,31800.0,51800.0,1SFCLA1UG,45X168.62 IRR,...,None,NaN,NaN,454.854546,7581.960830,"MULTIPOLYGON (((341022.808 150530.635, 340988....",COM3,1.0,None,None
2,0602_256_4.01,15F,CUMBERLAND,1723 MAIN STREET,0602_256_4.01,18000.0,66400.0,84400.0,2SFCLA,35X174.87 IRR,...,None,NaN,NaN,445.761001,6566.822274,"MULTIPOLYGON (((340978.236 150536.824, 340952....",COM8,2.0,None,None
3,0602_70_277,2,CUMBERLAND,247 MIST ROAD,0602_70_277,31600.0,49900.0,81500.0,1SFVLM,80 X 100,...,5214,1.0,NaN,362.125209,8098.973558,"MULTIPOLYGON (((337582.678 182335.442, 337559....",RES1,1.0,None,None
4,0602_139_7902,15C,CUMBERLAND,BEAVER DR,0602_139_7902,3000.0,0.0,3000.0,None,40X100,...,None,NaN,NaN,271.939615,3771.512282,"MULTIPOLYGON (((337982.218 181371.2, 337889.23...",GOV1,NaN,None,None


Here we can calculate some statistics about what we just parsed out. We also note there are some problems we need to identify in the string parsing, especially with mobile homes. However, I think it best to finish the workflow fully and then come back to this.

In [35]:
# do the statistics

## Part 4 - Combine Parcels with NSI
Now, we can merge with the NSI data using the geometries. We will do a spatial join and find matching NSI points and footprints, combining parcels with 1 or more NSI points into one row and excluding NSI points with no matching parcel. Parcels with no matching NSI can be kept. Then, we will compare building types and see which match and flag those that don't. Ultimately, we will always prioritize the information from the parcel, but NSI is essential for things like FFE (which we did MC on) and for dealing with parcels where we got no information about foundation type, occtype and/or number of stories from the parcel data. 

We recognize there are cases where always prioritizing parcel data leads to data loss (e.g. NSI points that do not match the parcel type are actually correct and may even match OBM footprints but the parcel suggests the land use is different) but because we are using improvement values from the parcels instead of the NSI estimated replacement cost (which is based on an algorithm, already uncertain and additionally propogates even more uncertainty from the buildings) I think it is better to just drop these for consistency. 

For reference on the NSI valuations discussed above [Cox and Sanderson](https://www.sciencedirect.com/science/article/abs/pii/S2212420923002352#:~:text=The%20input%20attributes%20show%20large,hazard%20damage%20and%20loss%20modeling.) find that when used for earthquake risk in the Cascadia subduction zone, the NSI underestimates property value compared to tax assessor values when aggregated to tract level. However, Renato and I found all sorts of strange structures, especially on vacant or AGR1 land. We should really investigate the values of AGR1 points in NSI in more detail as they seem to be frequently misclassified despite the presence of no structures, but I am not sure using the parcel data which routinely sets them to 0 when small structures are present on the farm is the best approach. Nonetheless, underestimating these losses seems preferable to $5 million + points systematically appearing on vacant land like we see in NSI.

In [36]:
nsi_nj_clean.crs  # chck the geometries match

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [37]:
parcels_nj_parsed.crs

<Projected CRS: EPSG:3424>
Name: NAD83 / New Jersey (ftUS)
Axis Info [cartesian]:
- X[east]: Easting (US survey foot)
- Y[north]: Northing (US survey foot)
Area of Use:
- name: United States (USA) - New Jersey - counties of Atlantic; Bergen; Burlington; Camden; Cape May; Cumberland; Essex; Gloucester; Hudson; Hunterdon; Mercer; Middlesex; Monmouth; Morris; Ocean; Passaic; Salem; Somerset; Sussex; Union; Warren.
- bounds: (-75.6, 38.87, -73.88, 41.36)
Coordinate Operation:
- name: SPCS83 New Jersey zone (US survey foot)
- method: Transverse Mercator
Datum: North American Datum 1983
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [38]:
## need to fix crs to for matching
if nsi_nj_clean.crs != parcels_nj_parsed.crs:
        parcels_nj_parsed = parcels_nj_parsed.to_crs(nsi_nj_clean.crs)

In [39]:
# we will only keep these columns, others are bloat
parcel_cols = [
    'GIS_PIN','PROP_CLASS', 'BLDG_DESC', 'PARSED_OCCTYPE', 'PARSED_STORIES', 
    'PARSED_FOUNDATION','LAND_VAL', 'IMPRVT_VAL', 'NET_VALUE', 'DWELL',
    'geometry']

nsi_cols = [
    'fd_id','cbfips','occtype','num_story', 'ffe_q05', 'ffe_q25', 'ffe_q50',
    'ffe_q75', 'ffe_q95', 'found_type','found_ht','val_struct','val_cont','geometry']


In [40]:
def join_nsi_to_parcels(nsi_gdf, parcels_gdf):
    """
    Spatial join NSI to parcels, preserving parcel geometry.

    We need to keep the parcel geometry so that we can deal 
    with the footprints later.

    Parameters
    ----------
    nsi_gdf : gpd.gdf
        geodataframe containing the NSI points
    parcels_gdf : gpd.gdf
        geodataframe containing the parcel geometries
    """
    # Rename geometries to keep both
    nsi = nsi_gdf.copy()
    parcels = parcels_gdf.copy()
    
    nsi['nsi_geom'] = nsi.geometry
    parcels['parcel_geom'] = parcels.geometry
    
    # Spatial join
    joined = gpd.sjoin(
        nsi,
        parcels,
        how='left',
        predicate='within'
    )
    
    joined = joined.drop(columns=['index_right'], errors='ignore')
    
    print(f"Total NSI points: {len(nsi)}")
    print(f"NSI matched to parcel: {joined['GIS_PIN'].notna().sum()}")
    print(f"NSI with no parcel: {joined['GIS_PIN'].isna().sum()}")
    
    return joined


In [41]:
# rename the geometries to be different
parcels = parcels_nj_parsed[parcel_cols].copy()
nsi = nsi_nj_clean[nsi_cols].copy()

# join them toegther
joined = join_nsi_to_parcels(nsi, parcels)

Total NSI points: 2766189
NSI matched to parcel: 2779573
NSI with no parcel: 75145


In [42]:
joined.head(5)

,fd_id,cbfips,occtype,num_story,ffe_q05,ffe_q25,ffe_q50,ffe_q75,ffe_q95,found_type,...,PROP_CLASS,BLDG_DESC,PARSED_OCCTYPE,PARSED_STORIES,PARSED_FOUNDATION,LAND_VAL,IMPRVT_VAL,NET_VALUE,DWELL,parcel_geom
0,548509648,340297280001017,RES1,2.0,0.53,1.22,1.75,2.43,3.33,C,...,2,2SF1G 2584,RES1,2.0,None,432800.0,268200.0,701000.0,1.0,"MULTIPOLYGON (((-74.08466 39.91144, -74.08493 ..."
1,548511068,340297131001010,RES1,1.0,0.22,0.44,0.64,0.88,1.22,B,...,2,1SF2G 1160,RES1,1.0,None,133200.0,89000.0,222200.0,1.0,"POLYGON ((-74.10464 40.10043, -74.10464 40.100..."
2,548523256,340410311013002,RES1,1.0,0.21,0.44,0.66,0.89,1.22,B,...,3B,None,AGR1,NaN,None,66600.0,0.0,66600.0,1.0,"MULTIPOLYGON (((-75.02765 40.96072, -75.03303 ..."
3,548545953,340297310021035,RES1,2.0,0.19,0.43,0.63,0.88,1.22,B,...,2,1SF1G 1192,RES1,1.0,None,100000.0,112100.0,212100.0,1.0,"POLYGON ((-74.14957 39.89436, -74.14949 39.894..."
4,548549026,340297140004002,RES1,1.0,0.19,0.42,0.63,0.88,1.24,B,...,2,1SF1G 1430,RES1,1.0,None,130000.0,120200.0,250200.0,1.0,"POLYGON ((-74.15317 40.0467, -74.15312 40.0466..."


In [43]:
# now we can go through and find all the NSI points that match a parcel
matched = joined[joined['GIS_PIN'].notna()].copy()  # right now this could have double counting, but that should get addressed later, regardless removes NSI with no match

vacant_nsi = matched[matched['PROP_CLASS'] == '1']
print(f"\nNSI points on vacant lots: {len(vacant_nsi)}")
print(f"Vacant lot NSI occtypes: {vacant_nsi['occtype'].value_counts().to_dict()}")
print(f'Max parcel improvement value: USD {vacant_nsi['IMPRVT_VAL'].max()}')  # check to ensure there is not an actual property masquerading as a vacant lot
vacant_nsi.head(5)


NSI points on vacant lots: 16616
Vacant lot NSI occtypes: {'RES1': 10021, 'COM4': 1463, 'GOV1': 892, 'RES3A': 460, 'COM1': 381, 'COM3': 377, 'RES3B': 322, 'COM8': 312, 'IND2': 296, 'COM2': 270, 'AGR1': 239, 'IND6': 227, 'REL1': 202, 'RES3C': 201, 'COM7': 157, 'RES2': 117, 'RES3D': 101, 'COM5': 94, 'EDU1': 86, 'RES3E': 68, 'IND1': 66, 'RES3F': 63, 'RES4': 56, 'GOV2': 26, 'RES6': 26, 'IND3': 21, 'COM10': 19, 'IND5': 18, 'COM6': 17, 'IND4': 10, 'RES5': 5, 'COM9': 2, 'EDU2': 1}
Max parcel improvement value: USD 0.0


,fd_id,cbfips,occtype,num_story,ffe_q05,ffe_q25,ffe_q50,ffe_q75,ffe_q95,found_type,...,PROP_CLASS,BLDG_DESC,PARSED_OCCTYPE,PARSED_STORIES,PARSED_FOUNDATION,LAND_VAL,IMPRVT_VAL,NET_VALUE,DWELL,parcel_geom
41,548905448,340190113032006,RES1,3.0,0.19,0.44,0.64,0.90,1.22,B,...,1,OPEN SPACE,None,NaN,None,0.0,0.0,0.0,1.0,"MULTIPOLYGON (((-74.83117 40.49169, -74.83112 ..."
53,549079910,340390387006004,RES3D,1.0,0.20,0.44,0.63,0.88,1.22,B,...,1,None,None,NaN,None,731800.0,0.0,731800.0,0.0,"MULTIPOLYGON (((-74.38297 40.63978, -74.38276 ..."
163,550637732,340057038031008,RES1,2.0,0.18,0.45,0.64,0.89,1.23,B,...,1,BIRCHWOOD LAKES,None,NaN,None,500.0,0.0,500.0,NaN,"MULTIPOLYGON (((-74.82979 39.86997, -74.8299 3..."
264,548473409,340330219001007,COM4,3.0,0.53,1.19,1.74,2.37,3.26,S,...,1,None,None,NaN,None,233300.0,0.0,233300.0,NaN,"MULTIPOLYGON (((-75.46673 39.57814, -75.46643 ..."
518,548445610,340330206002082,COM8,1.0,0.53,1.21,1.74,2.44,3.28,S,...,1,None,RES4,NaN,None,1175000.0,0.0,1175000.0,NaN,"MULTIPOLYGON (((-75.47308 39.68301, -75.47323 ..."


In [44]:
matched = matched[matched['PROP_CLASS'] != '1']  # dropping the vacant land (there is no curve for this plus they all have imprvt value of 0 anyway)

# remove rows with missing PROP_CLASS in the parcel data (note that when i checked this there were none after the join)
matched = matched[matched['PROP_CLASS'].notna()]
matched = matched[matched['PROP_CLASS'] != '']
print(f"\nAfter filtering: {len(matched)} matched points (including duplicates for now)")


After filtering: 2762957 matched points (including duplicates for now)


Here is some verification that the occupancy type data is fairly clean:

In [45]:
matched.PARSED_OCCTYPE.unique()

array(['RES1', 'AGR1', 'RES2', 'RES3A', 'COM4', 'COM1', 'IND1', 'GOV1',
       'COM8', 'REL1', 'RES3C', 'COM10', 'RES4', 'RES5', 'COM3', 'RES3B',
       'EDU1', 'IND2', 'RES3D', 'IND3', 'COM5', 'COM7', 'IND6', 'RES6',
       'IND4', 'COM6', 'COM2', 'RES3E', 'COM9', 'GOV2', 'EDU2', 'RES3F'],
      dtype=object)

In [46]:
len(parcels_nj_clean)

3291889

In [47]:
missing_prop = parcels_nj_clean["PROP_CLASS"].isna()
missing_inferred = parcels_nj_clean["PARSED_OCCTYPE"].isna()

both_missing = missing_prop & missing_inferred
prop_only = missing_prop & ~missing_inferred
inferred_only = ~missing_prop & missing_inferred

total_missing_prop = missing_prop.sum()
total_missing_inferred = missing_inferred.sum()
missing_both = both_missing.sum()
missing_inferred_only = inferred_only.sum()

total = len(parcels_nj_clean)

print(f"Total buildings: {total:,}\n")

print(
    f"Missing PROP_CLASS: {total_missing_prop:,} "
    f"({total_missing_prop / total * 100:.2f}%)"
)
print(
    f"Missing inferred occupancy (PARSED_OCCTYPE): {total_missing_inferred:,} "
    f"({total_missing_inferred / total * 100:.2f}%)"
)
print(
    f"Missing inferred occupancy because PROP_CLASS is missing (both missing): "
    f"{missing_both:,} ({missing_both / total * 100:.2f}%)"
)
print(
    f"Have PROP_CLASS but missing inferred occupancy: "
    f"{missing_inferred_only:,} ({missing_inferred_only / total * 100:.2f}%)"
)


Total buildings: 3,291,889

Missing PROP_CLASS: 227,847 (6.92%)
Missing inferred occupancy (PARSED_OCCTYPE): 374,635 (11.38%)
Missing inferred occupancy because PROP_CLASS is missing (both missing): 227,847 (6.92%)
Have PROP_CLASS but missing inferred occupancy: 146,788 (4.46%)


In [48]:
subset_missing_ot = parcels_nj_clean.loc[inferred_only]
print(f'unique improvement values: {subset_missing_ot.IMPRVT_VAL.unique()}')
print(f'unique PROP_CLASS: {subset_missing_ot.PROP_CLASS.unique()}')

unique improvement values: [     0. 477200.]
unique PROP_CLASS: ['1']


Fortunately, all the data we couldn't parse *something* out of the parcels for turns out to be vacant land, which we already excluded. No need to worry about this going forward! 

The final merge is completed here - in all cases we let the data we parse out of the parcels superceede the NSI. I have found some cases in QGIS where this may not be entirely valid, especially because at this moment there is a year mismatch between the parcels (2024 tax year) and the NSI (2022, though we don't really know when the survey was done...) I requested the 2021 Tax Year MOD-IV parcels geodatabase from the Rutgers portal, but this should be sufficient for now at least.

In [49]:
def final_occtype(parcel_occ, nsi_occ):
    """
    Parcel data takes priority if available.
    
    Note that this shouldn't really happen as the 
    parcels where occtype couldnt be infered should 
    have already been dropped as we showed above. I am
    adding this function mostly for use on the national 
    scale.

    Parameters
    -----------
    parcel_occ : string
        the occtype we parsed from the parcel data
    nsi_occ : string
        the occtype stated in NSI
    """
    if pd.notna(parcel_occ) and parcel_occ != '':
        return parcel_occ
    return nsi_occ

## Part 5 - Some manual adjustments based on street view surveys
It is quicker and easier to just "manually" fix the few annoying story types where the assessor entered the square footage into the building description in such a way that the pattern matches the standard classification code but is actually square feet (e.g. 1437SF/T.H. would imply 1437 stories by the MOD-IV doccumentation but this is pretty clearly square feet...). There are also a few which when checked on street view clearly have 1 story and the building type has just been attached to the BLDG_DESC field by the assessor (e.g. 21SFR2G for a couple of properties in downtown Princeton). I can come back and add examples of this for documentation later. 

In [50]:
with np.printoptions(suppress=True, precision=4):
    print(np.sort(matched[matched.PROP_CLASS == '2'].PARSED_STORIES.unique()))

[ -0.5    0.     0.5    1.     1.15   1.2    1.25   1.34   1.5    1.52
   1.55   1.6    1.7    1.75   2.     2.1    2.2    2.5    2.55   2.7
   2.75   3.     3.5    4.     4.5    5.     5.5    6.     6.5    7.
   8.     9.    10.    11.    11.5   12.    13.    14.    15.    17.
  18.    19.    20.    21.    22.    23.    25.    25.5   26.    28.
  29.    31.    34.    50.    55.    62.5   79.    90.   105.   111.
 115.   131.   134.   152.   185.   195.   205.   240.   295.   342.
    nan]


almost all of the 15 story properties are misclassified from 1.5 stories when checked on street view. We will go back in and correct these. i also checked all of the RES3 reclassifications (ones which had 2 dwellings listed) against Zillow and the street view image to see if this was a potential error. Note that we cannot be 100% certain that the property isn't being rented as multi unit or something like that, but the damages are probably closer to RES1 in these cases.

In [51]:
matched[matched.PARSED_STORIES == 15].head(5)

,fd_id,cbfips,occtype,num_story,ffe_q05,ffe_q25,ffe_q50,ffe_q75,ffe_q95,found_type,...,PROP_CLASS,BLDG_DESC,PARSED_OCCTYPE,PARSED_STORIES,PARSED_FOUNDATION,LAND_VAL,IMPRVT_VAL,NET_VALUE,DWELL,parcel_geom
33398,548478536,340297158001030,REL1,1.0,0.60,1.24,1.80,2.43,3.32,S,...,15D,15SSO 4688,REL1,15.0,None,265000.0,1034400.0,1299400.0,1.0,"MULTIPOLYGON (((-74.21583 40.07211, -74.21586 ..."
127785,548572894,340297160002045,RES1,1.0,0.19,0.44,0.64,0.88,1.21,B,...,2,15SAL&2AG 3044,RES1,15.0,None,81800.0,283200.0,365000.0,1.0,"MULTIPOLYGON (((-74.17625 40.03311, -74.17641 ..."
155131,548600270,340297158001066,EDU1,2.0,0.56,1.21,1.70,2.43,3.30,S,...,15B,"15SCB, 40645724",EDU1,15.0,None,600600.0,5508100.0,6108700.0,1.0,"MULTIPOLYGON (((-74.19482 40.05224, -74.19498 ..."
160531,548605682,340090213004040,RES1,2.0,0.20,0.45,0.64,0.89,1.23,B,...,2,15S-F,RES3A,15.0,None,250000.0,157600.0,407600.0,2.0,"MULTIPOLYGON (((-74.79763 39.00096, -74.79753 ..."
191618,548636794,340297153012007,RES1,3.0,0.19,0.42,0.62,0.86,1.22,B,...,2,15S-AL-F 4443,RES1,15.0,None,214500.0,290100.0,504600.0,1.0,"MULTIPOLYGON (((-74.21173 40.10398, -74.21219 ..."


In [52]:
mask = (matched['PARSED_STORIES'] == 15) & (matched['PROP_CLASS'] == '2')

matched.loc[mask, 'PARSED_OCCTYPE'] = 'RES1'
matched.loc[mask, 'PARSED_STORIES'] = 1.5

matched[matched.PARSED_STORIES == 15].head(5)

,fd_id,cbfips,occtype,num_story,ffe_q05,ffe_q25,ffe_q50,ffe_q75,ffe_q95,found_type,...,PROP_CLASS,BLDG_DESC,PARSED_OCCTYPE,PARSED_STORIES,PARSED_FOUNDATION,LAND_VAL,IMPRVT_VAL,NET_VALUE,DWELL,parcel_geom
33398,548478536,340297158001030,REL1,1.0,0.60,1.24,1.80,2.43,3.32,S,...,15D,15SSO 4688,REL1,15.0,None,265000.0,1034400.0,1299400.0,1.0,"MULTIPOLYGON (((-74.21583 40.07211, -74.21586 ..."
155131,548600270,340297158001066,EDU1,2.0,0.56,1.21,1.70,2.43,3.30,S,...,15B,"15SCB, 40645724",EDU1,15.0,None,600600.0,5508100.0,6108700.0,1.0,"MULTIPOLYGON (((-74.19482 40.05224, -74.19498 ..."
330844,548776047,340170075003004,RES3A,1.0,0.21,0.43,0.64,0.88,1.22,B,...,4C,15S-255U-C-G,RES3D,15.0,None,22950000.0,82050000.0,105000000.0,NaN,"MULTIPOLYGON (((-74.04191 40.71494, -74.04129 ..."
335060,548780328,340170031003000,COM7,1.0,0.61,1.25,1.78,2.40,3.26,S,...,15F,15S-B-110U,GOV1,15.0,None,868300.0,0.0,868300.0,NaN,"MULTIPOLYGON (((-74.06233 40.72241, -74.06256 ..."
335061,548780329,340170031003000,REL1,1.0,0.57,1.23,1.76,2.43,3.27,S,...,15F,15S-B-110U,GOV1,15.0,None,868300.0,0.0,868300.0,NaN,"MULTIPOLYGON (((-74.06233 40.72241, -74.06256 ..."


We can see from the GIS_PIN being very close in value and the same, non-standard building code that this is all the same assessor. This looks like some filler for a region that was not assessed properly as manually checking them has not shown any particular pattern in classification (some have 2 stories, some have 1) and the incorrectly classified buildings are not consistent with NSI. For simplicty, we will just default these to 1 story, RES1 as this is the predominant building type here, but it might be worth coming back and actually doing these manually.

In [53]:
mask = (matched['PARSED_STORIES'] == 10) & (matched['PROP_CLASS'] == '2')

matched.loc[mask, 'PARSED_OCCTYPE'] = 'RES1'
matched.loc[mask, 'PARSED_STORIES'] = 1.0

matched[matched.PARSED_STORIES == 10].head(5)

,fd_id,cbfips,occtype,num_story,ffe_q05,ffe_q25,ffe_q50,ffe_q75,ffe_q95,found_type,...,PROP_CLASS,BLDG_DESC,PARSED_OCCTYPE,PARSED_STORIES,PARSED_FOUNDATION,LAND_VAL,IMPRVT_VAL,NET_VALUE,DWELL,parcel_geom
228348,548673457,340110403002003,RES6,1.0,0.52,1.23,1.76,2.43,3.26,S,...,15C,10S-CB-X,GOV1,10.0,None,38700.0,6748400.0,6787100.0,NaN,"MULTIPOLYGON (((-75.0099 39.4864, -75.00993 39..."
228352,548673461,340110403002003,GOV1,1.0,0.54,1.23,1.78,2.40,3.23,S,...,15C,10S-CB-X,GOV1,10.0,None,42600.0,6217200.0,6259800.0,NaN,"MULTIPOLYGON (((-75.00978 39.48735, -75.0099 3..."
228373,548673482,340110403002003,GOV1,1.0,0.56,1.21,1.77,2.38,3.31,S,...,15C,10S-CB-X,GOV1,10.0,None,42600.0,6217200.0,6259800.0,NaN,"MULTIPOLYGON (((-75.00978 39.48735, -75.0099 3..."
330920,548776153,340170074001004,COM4,1.0,0.61,1.23,1.77,2.41,3.27,C,...,4A,10S-258-RM HOTE,RES4,10.0,None,4037000.0,0.0,4037000.0,NaN,"MULTIPOLYGON (((-74.03396 40.71615, -74.03368 ..."
330921,548776154,340170074001004,IND6,1.0,0.57,1.22,1.78,2.44,3.26,C,...,4A,10S-258-RM HOTE,RES4,10.0,None,4037000.0,0.0,4037000.0,NaN,"MULTIPOLYGON (((-74.03396 40.71615, -74.03368 ..."


It appears that this surveyor has included the PROP_CLASS (2 for residential) ahead of the actual buidling description ('1SFR2G').

In [54]:
matched[matched.PARSED_STORIES == 21]

,fd_id,cbfips,occtype,num_story,ffe_q05,ffe_q25,ffe_q50,ffe_q75,ffe_q95,found_type,...,PROP_CLASS,BLDG_DESC,PARSED_OCCTYPE,PARSED_STORIES,PARSED_FOUNDATION,LAND_VAL,IMPRVT_VAL,NET_VALUE,DWELL,parcel_geom
34840,548479978,340297140001000,RES1,1.0,0.21,0.43,0.63,0.88,1.23,B,...,2,21SF2G 2115,RES1,21.0,None,123400.0,160200.0,283600.0,1.0,"MULTIPOLYGON (((-74.15057 40.05469, -74.15065 ..."
48902,548494040,340297280003042,COM1,1.0,6.93,8.08,8.96,9.89,10.97,I,...,4A,21SCB 4978,RES1,21.0,None,550000.0,164200.0,714200.0,1.0,"MULTIPOLYGON (((-74.07672 39.92932, -74.07676 ..."
50423,548495562,340297280003037,RES1,2.0,6.95,8.15,9.00,9.85,10.99,I,...,2,21SCB 2688,RES3A,21.0,None,400000.0,235000.0,635000.0,2.0,"MULTIPOLYGON (((-74.07841 39.9292, -74.07845 3..."
59495,548504638,340297280003041,RES1,1.0,6.93,8.10,8.98,9.84,11.03,I,...,2,21SF 0936,RES3A,21.0,None,559100.0,151900.0,711000.0,2.0,"MULTIPOLYGON (((-74.07541 39.92965, -74.07546 ..."
365141,548810321,340170075001002,RES3A,1.0,0.20,0.42,0.63,0.90,1.24,B,...,4C,21S-400U-COOP,RES5,21.0,None,25545000.0,21353300.0,46898300.0,NaN,"MULTIPOLYGON (((-74.0404 40.71777, -74.04051 4..."
365142,548810322,340170075001002,RES3A,1.0,0.19,0.44,0.62,0.88,1.21,B,...,4C,21S-C-390U-COOP,RES5,21.0,None,25025000.0,20495800.0,45520800.0,NaN,"MULTIPOLYGON (((-74.04051 40.71778, -74.0404 4..."
367454,548812630,340170075004000,RES3A,1.0,0.53,1.20,1.71,2.36,3.23,C,...,4C,21S-200U-COOP,RES5,21.0,None,18000000.0,37713000.0,55713000.0,NaN,"MULTIPOLYGON (((-74.0412 40.71652, -74.04109 4..."
367480,548812656,340170077003004,COM5,1.0,0.53,1.19,1.79,2.42,3.26,S,...,4A,21S-ST-O-HVAC-E,COM1,21.0,None,29718800.0,164320400.0,194039200.0,NaN,"MULTIPOLYGON (((-74.0348 40.72817, -74.03475 4..."
367481,548812657,340170077003004,COM4,3.0,0.57,1.21,1.74,2.37,3.25,S,...,4A,21S-ST-O-HVAC-E,COM1,21.0,None,29718800.0,164320400.0,194039200.0,NaN,"MULTIPOLYGON (((-74.0348 40.72817, -74.03475 4..."
568001,549013243,340210042012002,RES1,1.0,0.20,0.43,0.64,0.89,1.23,B,...,2,21SFR2G,RES1,21.0,None,472000.0,274400.0,746400.0,1.0,"MULTIPOLYGON (((-74.63718 40.36733, -74.63692 ..."


In [55]:
mask = (matched['PARSED_STORIES'] == 21) & (matched['PROP_CLASS'] == '2')

matched.loc[mask, 'PARSED_OCCTYPE'] = 'RES1'
matched.loc[mask, 'PARSED_STORIES'] = 1.0

matched[matched.PARSED_STORIES == 10].head(5)

,fd_id,cbfips,occtype,num_story,ffe_q05,ffe_q25,ffe_q50,ffe_q75,ffe_q95,found_type,...,PROP_CLASS,BLDG_DESC,PARSED_OCCTYPE,PARSED_STORIES,PARSED_FOUNDATION,LAND_VAL,IMPRVT_VAL,NET_VALUE,DWELL,parcel_geom
228348,548673457,340110403002003,RES6,1.0,0.52,1.23,1.76,2.43,3.26,S,...,15C,10S-CB-X,GOV1,10.0,None,38700.0,6748400.0,6787100.0,NaN,"MULTIPOLYGON (((-75.0099 39.4864, -75.00993 39..."
228352,548673461,340110403002003,GOV1,1.0,0.54,1.23,1.78,2.40,3.23,S,...,15C,10S-CB-X,GOV1,10.0,None,42600.0,6217200.0,6259800.0,NaN,"MULTIPOLYGON (((-75.00978 39.48735, -75.0099 3..."
228373,548673482,340110403002003,GOV1,1.0,0.56,1.21,1.77,2.38,3.31,S,...,15C,10S-CB-X,GOV1,10.0,None,42600.0,6217200.0,6259800.0,NaN,"MULTIPOLYGON (((-75.00978 39.48735, -75.0099 3..."
330920,548776153,340170074001004,COM4,1.0,0.61,1.23,1.77,2.41,3.27,C,...,4A,10S-258-RM HOTE,RES4,10.0,None,4037000.0,0.0,4037000.0,NaN,"MULTIPOLYGON (((-74.03396 40.71615, -74.03368 ..."
330921,548776154,340170074001004,IND6,1.0,0.57,1.22,1.78,2.44,3.26,C,...,4A,10S-258-RM HOTE,RES4,10.0,None,4037000.0,0.0,4037000.0,NaN,"MULTIPOLYGON (((-74.03396 40.71615, -74.03368 ..."


For buildings with over 1000 parsed stories, we can see that the assessor has put the square footage where the building description should be. There are probably too many of these to deal with manually for now, so we will just revert their stories to whatever is listed in NSI. We should really try to parse out what we have in some of the BLDG_DESC (e.g. colonial and so on) by adding some max stories catch in the regex parsing function... 

## Part 6 - Final Cleaning
In this section we will do some final cleaning of the merged NSI and parcel data. Using the available information, we will decide what the final types we have for each point in the parcel, which will be the data we merge with the OBM footprints and more importantly derive our HAZUS DDF from.

In [56]:
def final_stories(parcel_stories, nsi_stories):
    """
    Determines the final number of stories we 
    use for a given entry. 
    
    Parcel data takes priority if available.
    Round 2.5 up to 3.

    Parameters
    -----------
    parcel_stories : float
        the number of stories we got from the parcel
    nsi_stories : float
        the number of stories given in the NSI 

    Returns
    --------
    stories : float
        the final decision on number of stories (parcel_stories unless None)
    """
    stories = parcel_stories if pd.notna(parcel_stories) else nsi_stories
    if pd.isna(stories):
        return None
    # Round 2.5 -> 3, otherwise keep as is
    if 1.0 < stories < 1.5:
        return 1.0
    if 1.5 < stories < 2.0:
        return 2.0
    if stories >= 2.5:
        return 3.0
    return stories

In [57]:
def final_foundation(parcel_found, nsi_found):
    """Parcel data takes priority if available."""
    if pd.notna(parcel_found) and parcel_found != '':
        return parcel_found
    return nsi_found


In [58]:
def has_basement(foundation):
    """
    Determine if foundation indicates basement.

    Note that when we initially merged with NSI 
    we already dealt with a "final basement" judgement,
    so we will just pluck from that.

    Parameters
    ----------
    foundation : string
        the foundation type we determined in the merged dat

    Returns
    --------
        : Bool
        is there a basement foundation or not?
    """
    if pd.isna(foundation):
        return False
    foundation = str(foundation).upper()
    if foundation in ['BASEMENT', 'B']:
        return True
    if foundation in ['SLAB', 'S', 'CRAWL', 'C']:
        return False
    return False

In [59]:
## these are basically data quality flagging functions to check whether our data
## is matching the NSI well. This could probably be used to fix things like the 
## misclassification of mobile homes etc but I am not sure we have the time for 
## that and we are really only trying to get a rough estimate of damages...

def compare_occtype(parcel_occ, nsi_occ):
    """
    Compare occupancy types, return match status.

    Parameters
    ----------
    parcel_occ : string
        the occupancy type from the parcel data
    nsi_occ : string
        the occupancy type from the NSI data

    Returns
    -------
        : string
        a data quality flag for that building/parcel
    """
    if pd.isna(parcel_occ) or pd.isna(nsi_occ):
        return 'MISSING'
    if parcel_occ == nsi_occ:
        return 'MATCH'
    # Check if same broad category (e.g., both RES)
    if parcel_occ[:3] == nsi_occ[:3]:
        return 'PARTIAL'  # Same category, different subtype
    return 'MISMATCH'

def compare_stories(parcel_stories, nsi_stories):
    """
    Compare number of stories, return match status.

    Parameters
    ----------
    parcel_stories : string
        the stories from the parcel data
    nsi_stories : string
        the stories from the NSI data

    Returns
    -------
        : string
        a data quality flag for that building/parcel
    """
    if pd.isna(parcel_stories) or pd.isna(nsi_stories):
        return 'MISSING'
    if parcel_stories == nsi_stories:
        return 'MATCH'
    if abs(parcel_stories - nsi_stories) <= 0.5:
        return 'CLOSE'  # Within 0.5 stories
    return 'MISMATCH'

# add data quality flags to the matched dataframes and see how it looks
matched['occtype_compare'] = matched.apply(
    lambda r: compare_occtype(r['PARSED_OCCTYPE'], r['occtype']), axis=1
)

matched['stories_compare'] = matched.apply(
    lambda r: compare_stories(r['PARSED_STORIES'], r['num_story']), axis=1
)

print("\nOcctype comparison:")
print(matched['occtype_compare'].value_counts())

print("\nStories comparison:")
print(matched['stories_compare'].value_counts())


Occtype comparison:
occtype_compare
MATCH       1975549
PARTIAL      608195
MISMATCH     179213
Name: count, dtype: int64

Stories comparison:
stories_compare
MISSING     1232176
MATCH        913804
MISMATCH     482372
CLOSE        134605
Name: count, dtype: int64


In [60]:
# now apply the final parsing
matched['final_occtype'] = matched.apply(
    lambda r: final_occtype(r['PARSED_OCCTYPE'], r['occtype']), axis=1
)

matched['final_stories'] = matched.apply(
    lambda r: final_stories(r['PARSED_STORIES'], r['num_story']), axis=1
)

matched['final_foundation'] = matched.apply(
    lambda r: final_foundation(r['PARSED_FOUNDATION'], r['found_type']), axis=1
)

matched['has_basement'] = matched['final_foundation'].apply(has_basement)

In [61]:
matched.head(5)   # take a look at the data to see check it looks how we expected

,fd_id,cbfips,occtype,num_story,ffe_q05,ffe_q25,ffe_q50,ffe_q75,ffe_q95,found_type,...,IMPRVT_VAL,NET_VALUE,DWELL,parcel_geom,occtype_compare,stories_compare,final_occtype,final_stories,final_foundation,has_basement
0,548509648,340297280001017,RES1,2.0,0.53,1.22,1.75,2.43,3.33,C,...,268200.0,701000.0,1.0,"MULTIPOLYGON (((-74.08466 39.91144, -74.08493 ...",MATCH,MATCH,RES1,2.0,C,False
1,548511068,340297131001010,RES1,1.0,0.22,0.44,0.64,0.88,1.22,B,...,89000.0,222200.0,1.0,"POLYGON ((-74.10464 40.10043, -74.10464 40.100...",MATCH,MATCH,RES1,1.0,B,True
2,548523256,340410311013002,RES1,1.0,0.21,0.44,0.66,0.89,1.22,B,...,0.0,66600.0,1.0,"MULTIPOLYGON (((-75.02765 40.96072, -75.03303 ...",MISMATCH,MISSING,AGR1,1.0,B,True
3,548545953,340297310021035,RES1,2.0,0.19,0.43,0.63,0.88,1.22,B,...,112100.0,212100.0,1.0,"POLYGON ((-74.14957 39.89436, -74.14949 39.894...",MATCH,MISMATCH,RES1,1.0,B,True
4,548549026,340297140004002,RES1,1.0,0.19,0.42,0.63,0.88,1.24,B,...,120200.0,250200.0,1.0,"POLYGON ((-74.15317 40.0467, -74.15312 40.0466...",MATCH,MATCH,RES1,1.0,B,True


This is the part where we deal with double counting etc. by aggregating the NSI points to the parcels. We also add some data quality flags to inform our data quality checks later.

In [62]:
def aggregate_nsi_to_parcel(group):
    """
    Aggregate multiple NSI points within a parcel.
    Returns a single row per parcel, preserving parcel geometry.

    Note: this is probably the place to handle the outlier 
    cases where we are still misclassifying in the future,
    especially if we use similar function for the CONUS
    buildings, where UQ matters much more...

    Parameters
    ----------
    group : arrayLike[str]
        a list of duplicate NSI points for a parcel

    Returns
    -------
        : pd.Series
        a Pandas series containing the final information for the 
        grouped data. This replaces the old line in the gdf.
    """
    result = {
        # Parcel attributes (same for all points in group)
        'GIS_PIN': group['GIS_PIN'].iloc[0],
        'PROP_CLASS': group['PROP_CLASS'].iloc[0],
        'PARSED_OCCTYPE': group['PARSED_OCCTYPE'].iloc[0],
        'PARSED_STORIES': group['PARSED_STORIES'].iloc[0],
        'PARSED_FOUNDATION': group['PARSED_FOUNDATION'].iloc[0],
        'LAND_VAL': group['LAND_VAL'].iloc[0],
        'IMPRVT_VAL': group['IMPRVT_VAL'].iloc[0],
        'NET_VALUE': group['NET_VALUE'].iloc[0],
        'parcel_geom': group['parcel_geom'].iloc[0],  # Keep parcel geometry
        
        # NSI counts
        'nsi_count': len(group),
        
        # NSI attributes
        'nsi_fd_ids': list(group['fd_id']),
        'cbfips': group['cbfips'].iloc[0],
        'nsi_occtype': group['occtype'].iloc[0],
        'nsi_stories': group['num_story'].iloc[0],
        'found_type': group['found_type'].iloc[0],
        'nsi_val_struct': group['val_struct'].sum(),
        'nsi_val_cont': group['val_cont'].sum(),
        
        # Foundation height quantiles
        'found_ht_q05': group['ffe_q05'].min(),
        'found_ht_q25': group['ffe_q25'].min(),
        'found_ht_q50': group['ffe_q50'].min(),
        'found_ht_q75': group['ffe_q75'].min(),
        'found_ht_q95': group['ffe_q95'].min(),
        
        # Final resolved attributes
        'final_occtype': group['final_occtype'].iloc[0],
        'final_stories': group['final_stories'].iloc[0],
        'final_foundation': group['final_foundation'].iloc[0],
        'has_basement': group['has_basement'].iloc[0],
        
        # Comparison flags
        'occtype_compare': group['occtype_compare'].iloc[0],
        'stories_compare': group['stories_compare'].iloc[0],
        
        # Conflict flags
        'nsi_occtype_conflict': group['occtype'].nunique() > 1,
        'nsi_stories_conflict': group['num_story'].nunique() > 1,
        'nsi_foundation_conflict': group['found_type'].nunique() > 1,
    }
    return pd.Series(result)

In [63]:
%%time
parcel_agg = matched.groupby('GIS_PIN').apply(aggregate_nsi_to_parcel).reset_index(drop=True)

print(f"\nAggregated parcels: {len(parcel_agg)}")
print(f"Parcels with multiple NSI: {(parcel_agg['nsi_count'] > 1).sum()}")
print(f"Parcels with NSI occtype conflict: {parcel_agg['nsi_occtype_conflict'].sum()}")
print(f"Parcels with NSI stories conflict: {parcel_agg['nsi_stories_conflict'].sum()}")
print(f"Parcels with NSI foundation conflict: {parcel_agg['nsi_foundation_conflict'].sum()}")

<timed exec>:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



Aggregated parcels: 2480307
Parcels with multiple NSI: 119671
Parcels with NSI occtype conflict: 83192
Parcels with NSI stories conflict: 25111
Parcels with NSI foundation conflict: 56164
CPU times: user 34min 24s, sys: 21 s, total: 34min 45s
Wall time: 34min 53s


In [64]:
parcel_agg.columns

Index(['GIS_PIN', 'PROP_CLASS', 'PARSED_OCCTYPE', 'PARSED_STORIES',
       'PARSED_FOUNDATION', 'LAND_VAL', 'IMPRVT_VAL', 'NET_VALUE',
       'parcel_geom', 'nsi_count', 'nsi_fd_ids', 'cbfips', 'nsi_occtype',
       'nsi_stories', 'found_type', 'nsi_val_struct', 'nsi_val_cont',
       'found_ht_q05', 'found_ht_q25', 'found_ht_q50', 'found_ht_q75',
       'found_ht_q95', 'final_occtype', 'final_stories', 'final_foundation',
       'has_basement', 'occtype_compare', 'stories_compare',
       'nsi_occtype_conflict', 'nsi_stories_conflict',
       'nsi_foundation_conflict'],
      dtype='object')

In [65]:
parcel_agg.head(5)

,GIS_PIN,PROP_CLASS,PARSED_OCCTYPE,PARSED_STORIES,PARSED_FOUNDATION,LAND_VAL,IMPRVT_VAL,NET_VALUE,parcel_geom,nsi_count,...,found_ht_q95,final_occtype,final_stories,final_foundation,has_basement,occtype_compare,stories_compare,nsi_occtype_conflict,nsi_stories_conflict,nsi_foundation_conflict
0,0101_1.01_1.01,2,RES1,2.0,None,70000.0,158500.0,228500.0,MULTIPOLYGON (((-74.50451784059386 39.44854274...,1,...,1.23,RES1,2.0,B,True,MATCH,MATCH,False,False,False
1,0101_1.01_1.02,2,RES1,2.0,None,70000.0,144000.0,214000.0,MULTIPOLYGON (((-74.50469688347265 39.44838879...,1,...,1.23,RES1,2.0,B,True,MATCH,MISMATCH,False,False,False
2,0101_1.01_10,2,RES1,2.0,None,70000.0,157600.0,227600.0,MULTIPOLYGON (((-74.50800816897657 39.44759156...,1,...,1.21,RES1,2.0,B,True,PARTIAL,MISMATCH,False,False,False
3,0101_1.01_11,2,RES1,2.0,None,70000.0,134700.0,204700.0,MULTIPOLYGON (((-74.50827179621473 39.44777489...,1,...,1.21,RES1,2.0,B,True,MATCH,MATCH,False,False,False
4,0101_1.01_13,2,RES1,2.0,None,70000.0,133400.0,203400.0,MULTIPOLYGON (((-74.50704518707917 39.44636866...,1,...,1.25,RES1,2.0,B,True,MATCH,MISMATCH,False,False,False


## Part 7 - Repositioning Buildings with OBM Parcels
Yay! This is the final data processing section (for now...) and is done to better improve the representation of building location on large parcels, parcels where there are multiple small buildings which we want to aggregate (e.g. a row townhouse with many small RES1s for tax purposes becoming a RES3-x depending on the number of parcels it intersects), or parcels where we need to break up some buildings (e.g. a 4C apartment complex with many buildings that NSI classified wrong). Note that the latter of these can be challenging and is the main data quality issue in this dataset.

See https://onlinelibrary.wiley.com/doi/full/10.1111/jfr3.12851 for reference on parcels, though note we need to be stricter with matching parcel and NSI occtypes because ZTRAX is not available anymore and thus our data quality is much lower. This probably leads to data loss and further underestimation but being conservative here prevents gross over-evaluation later.

There are two big adjustments that get made here:
1. If a footprint intersects multiple RES3 points (see why we did all that protection from reclassifying apartments earlier!) we redefine the subtype to account for the number of parcels it intersects, which we assume to be better representative of the number of units. There is probably a secondary control we could do on this which might help with the outliers where RES1 is still assigned somewhere in the messy regex chain, but we are getting nitpicky for a minority contribution to the the paper at this point. What this is most important for is row/townhouses.
2. Disagregate apartment complexes with misplaced NSI points or single lot specifications. In the case that there are lots of similar sized footprints on a lot, we will disaggreate the land value equally between them. This is probably the most frustrating one, because it ultimately is hard to tune with a common rule. I tried all sorts of stuff to deal with this, as well as tiny overlaps between parcels and buildings on other lots, but it was just too much to fix without a lot more manual inspection etc.

In [66]:
def relocate_buildings_with_footprints(parcel_agg, footprints_gdf, 
                                        similar_size_ratio=0.005,
                                        min_building_sqm=10,
                                        min_overlap_ratio=0.005):
    """
    Relocate buildings using OBM footprints. This function is big
    and gross and I will be fully open in that I used Claude to help
    a lot with this. Basically the process was to write a starter code,
    get Claude to fix the mess, then make lots of adjustments with 
    trial and error to capture as much of the intersection between the
    buildings and footprints as possible without a second, confusing
    reclassification. 
    
    The reason for this is that this is a study about parcels not 
    the NSI/OBM. Those are supplementary datasets which fill in 
    information the parcels do not provide. This is still better
    in most cases than just using parcel centroids, which is only
    done where there are multiple NSI structures to a parcel and 
    the OBM footprints are too similar in size to pick one as the 
    primary building.
    
    Parameters
    ----------
    parcel_agg : GeoDataFrame
        Aggregated parcel data with 'parcel_geom' column
    footprints_gdf : GeoDataFrame
        Building footprints
    similar_size_ratio : float
        For RES1 with multiple footprints, if secondary/primary > this ratio,
        treat as similar sizes and split values. Default 0.7.
    min_building_sqm : float
        Minimum footprint area in sqm. Smaller footprints filtered out. Default 30.
    min_overlap_ratio : float
        Minimum overlap (footprint area in parcel / total footprint area) to count
        as a valid intersection. Filters marginal overlaps. Default 0.05.
    
    Returns
    -------
    GeoDataFrame
        Buildings with relocated geometries
    """
    parcels = parcel_agg.copy()
    if 'parcel_geom' not in parcels.columns:
        raise ValueError("parcel_agg must have 'parcel_geom' column")
    
    parcels = gpd.GeoDataFrame(parcels, geometry='parcel_geom', crs=footprints_gdf.crs)
    
    # Prep footprints
    footprints = footprints_gdf.copy()
    footprints['fp_area'] = footprints.geometry.area
    footprints = footprints[footprints['fp_area'] >= min_building_sqm].copy()
    footprints['fp_id'] = footprints['id']
    footprints['fp_centroid'] = footprints.geometry.centroid
    
    # Spatial join footprints to parcels
    fp_to_parcels = gpd.sjoin(
        footprints,
        parcels[['GIS_PIN', 'parcel_geom']].set_geometry('parcel_geom'),
        how='inner',
        predicate='intersects'
    ).drop(columns=['index_right'], errors='ignore')
    
    # Merge parcel geometry for overlap calculation
    fp_to_parcels = fp_to_parcels.merge(
        parcels[['GIS_PIN', 'parcel_geom']],
        on='GIS_PIN',
        how='left',
        suffixes=('', '_parcel')
    )
    
    # Count relationships
    fp_to_parcels['fp_parcel_count'] = fp_to_parcels.groupby('fp_id')['GIS_PIN'].transform('nunique')
    fp_to_parcels['parcel_fp_count'] = fp_to_parcels.groupby('GIS_PIN')['fp_id'].transform('nunique')
    
    # Rank footprints within each parcel by area
    fp_to_parcels['fp_rank'] = fp_to_parcels.groupby('GIS_PIN')['fp_area'].rank(ascending=False)
    
    # Get max area per parcel for ratio calculation
    fp_to_parcels['max_fp_area'] = fp_to_parcels.groupby('GIS_PIN')['fp_area'].transform('max')
    fp_to_parcels['area_ratio'] = fp_to_parcels['fp_area'] / fp_to_parcels['max_fp_area']
    
    # Merge parcel attributes
    fp_to_parcels = fp_to_parcels.merge(
        parcels.drop(columns=['parcel_geom']),
        on='GIS_PIN',
        how='left'
    )
    
    # ---------------------------------------------------------------------
    # CLASSIFY EACH FOOTPRINT-PARCEL PAIR
    # ---------------------------------------------------------------------
    
    # Rule 3: Footprint spans multiple parcels
    rule3_mask = fp_to_parcels['fp_parcel_count'] > 1
    
    # Rule 2a: RES1, multiple footprints, this is largest (different sizes)
    rule2a_mask = (
        ~rule3_mask &
        (fp_to_parcels['final_occtype'] == 'RES1') &
        (fp_to_parcels['parcel_fp_count'] > 1) &
        (fp_to_parcels['fp_rank'] == 1) &
        (fp_to_parcels.groupby('GIS_PIN')['area_ratio'].transform(
            lambda x: x.iloc[1] if len(x) > 1 else 0) < similar_size_ratio)
    )
    
    # Rule 2b: RES1, multiple footprints, similar sizes - keep all
    rule2b_mask = (
        ~rule3_mask &
        (fp_to_parcels['final_occtype'] == 'RES1') &
        (fp_to_parcels['parcel_fp_count'] > 1) &
        ~rule2a_mask &
        (fp_to_parcels.groupby('GIS_PIN')['area_ratio'].transform(
            lambda x: x.iloc[1] if len(x) > 1 else 0) >= similar_size_ratio)
    )
    
    # Rule 1: Non-RES1, multiple footprints - keep all
    rule1_mask = (
        ~rule3_mask &
        (fp_to_parcels['final_occtype'] != 'RES1') &
        (fp_to_parcels['parcel_fp_count'] > 1)
    )
    
    # Rule 4: Single footprint per parcel
    rule4_mask = ~rule3_mask & (fp_to_parcels['parcel_fp_count'] == 1)
    
    # Assign rules
    fp_to_parcels['rule'] = None
    fp_to_parcels.loc[rule3_mask, 'rule'] = 'multi_parcel_footprint'
    fp_to_parcels.loc[rule2a_mask, 'rule'] = 'res1_largest_footprint'
    fp_to_parcels.loc[rule2b_mask, 'rule'] = 'res1_similar_sizes'
    fp_to_parcels.loc[rule1_mask, 'rule'] = 'non_res1_multi_footprint'
    fp_to_parcels.loc[rule4_mask, 'rule'] = 'single_footprint'
    
    # Drop footprints that don't meet any rule
    fp_to_parcels = fp_to_parcels[fp_to_parcels['rule'].notna()].copy()
    
    # ---------------------------------------------------------------------
    # HELPER: Adjust RES3 unit count based on parcels spanned
    # ---------------------------------------------------------------------
    def adjust_res3_occtype(occtype, n_parcels):
        """Adjust RES3x based on number of parcels spanned."""
        if not isinstance(occtype, str) or not occtype.startswith('RES3'):
            return occtype
        
        unit_ranges = {
            'RES3A': (2, 2),
            'RES3B': (3, 4),
            'RES3C': (5, 9),
            'RES3D': (10, 19),
            'RES3E': (20, 49),
            'RES3F': (50, 999),
        }
        
        min_units = n_parcels
        
        for occ, (low, high) in unit_ranges.items():
            if low <= min_units <= high:
                return occ
        
        if min_units >= 50:
            return 'RES3F'
        
        return occtype
    
    # ---------------------------------------------------------------------
    # BUILD OUTPUT
    # ---------------------------------------------------------------------
    
    output_rows = []
    processed_parcels = set()
    
    # Rule 3: Aggregate parcels per footprint
    rule3 = fp_to_parcels[fp_to_parcels['rule'] == 'multi_parcel_footprint']
    for fp_id, group in rule3.groupby('fp_id'):
        n_parcels = group['GIS_PIN'].nunique()
        base_occtype = group['final_occtype'].mode().iloc[0]
        adjusted_occtype = adjust_res3_occtype(base_occtype, n_parcels)
        
        output_rows.append({
            'GIS_PIN': ','.join(group['GIS_PIN'].unique().astype(str)),
            'nsi_val_struct': group.drop_duplicates('GIS_PIN')['nsi_val_struct'].sum(),  
            'nsi_val_cont': group.drop_duplicates('GIS_PIN')['nsi_val_cont'].sum(),      
            'LAND_VAL': group.drop_duplicates('GIS_PIN')['LAND_VAL'].sum(),
            'IMPRVT_VAL': group.drop_duplicates('GIS_PIN')['IMPRVT_VAL'].sum(),
            'NET_VALUE': group.drop_duplicates('GIS_PIN')['NET_VALUE'].sum(),
            'final_occtype': adjusted_occtype,
            'original_occtype': base_occtype,
            'parcels_spanned': n_parcels,
            'final_stories': group['final_stories'].median(),
            'final_foundation': group['final_foundation'].mode().iloc[0] if len(group['final_foundation'].mode()) > 0 else None,
            'has_basement': group['has_basement'].mode().iloc[0] if len(group['has_basement'].mode()) > 0 else None,
            'found_ht_q05': group['found_ht_q05'].min(),
            'found_ht_q25': group['found_ht_q25'].min(),
            'found_ht_q50': group['found_ht_q50'].min(),
            'found_ht_q75': group['found_ht_q75'].min(),
            'found_ht_q95': group['found_ht_q95'].min(),
            'nsi_count': group.drop_duplicates('GIS_PIN')['nsi_count'].sum(),
            'fp_id': fp_id,
            'relocation_rule': 'multi_parcel_footprint',
            'geometry': group['fp_centroid'].iloc[0],
        })
        processed_parcels.update(group['GIS_PIN'].unique())
    
    # Rules 1, 2a, 2b, 4: One row per footprint
    other_rules = fp_to_parcels[
        (fp_to_parcels['rule'] != 'multi_parcel_footprint') &
        (~fp_to_parcels['GIS_PIN'].isin(processed_parcels))
    ].copy()
    
    other_rules['n_fps_kept'] = other_rules.groupby('GIS_PIN')['fp_id'].transform('count')
    other_rules['val_divisor'] = np.where(other_rules['n_fps_kept'] > 1, other_rules['n_fps_kept'], 1)
    
    for _, row in other_rules.iterrows():
        output_rows.append({
            'GIS_PIN': row['GIS_PIN'],
            'nsi_val_struct': row['nsi_val_struct'] / row['val_divisor'],
            'nsi_val_cont': row['nsi_val_cont'] / row['val_divisor'], 
            'LAND_VAL': row['LAND_VAL'] / row['val_divisor'],
            'IMPRVT_VAL': row['IMPRVT_VAL'] / row['val_divisor'],
            'NET_VALUE': row['NET_VALUE'] / row['val_divisor'],
            'final_occtype': row['final_occtype'],
            'original_occtype': row['final_occtype'],
            'parcels_spanned': 1,
            'final_stories': row['final_stories'],
            'final_foundation': row['final_foundation'],
            'has_basement': row['has_basement'],
            'found_ht_q05': row['found_ht_q05'],
            'found_ht_q25': row['found_ht_q25'],
            'found_ht_q50': row['found_ht_q50'],
            'found_ht_q75': row['found_ht_q75'],
            'found_ht_q95': row['found_ht_q95'],
            'nsi_count': row['nsi_count'] / row['val_divisor'],
            'fp_id': row['fp_id'],
            'relocation_rule': row['rule'],
            'geometry': row['fp_centroid'],
        })
        processed_parcels.add(row['GIS_PIN'])
    
    # Parcels with no footprint - use parcel centroid
    no_fp_parcels = parcels[~parcels['GIS_PIN'].isin(processed_parcels)]
    for _, row in no_fp_parcels.iterrows():
        output_rows.append({
            'GIS_PIN': row['GIS_PIN'],
            'nsi_val_struct': row['nsi_val_struct'],
            'nsi_val_cont': row['nsi_val_cont'],
            'LAND_VAL': row['LAND_VAL'],
            'IMPRVT_VAL': row['IMPRVT_VAL'],
            'NET_VALUE': row['NET_VALUE'],
            'final_occtype': row['final_occtype'],
            'original_occtype': row['final_occtype'],
            'parcels_spanned': 1,
            'final_stories': row['final_stories'],
            'final_foundation': row['final_foundation'],
            'has_basement': row['has_basement'],
            'found_ht_q05': row['found_ht_q05'],
            'found_ht_q25': row['found_ht_q25'],
            'found_ht_q50': row['found_ht_q50'],
            'found_ht_q75': row['found_ht_q75'],
            'found_ht_q95': row['found_ht_q95'],
            'nsi_count': row['nsi_count'],
            'fp_id': None,
            'relocation_rule': 'parcel_centroid',
            'geometry': row['parcel_geom'].centroid,
        })
    
    # Create output
    output_gdf = gpd.GeoDataFrame(output_rows, crs=footprints_gdf.crs)
    
    # Summary
    print("\n" + "="*50)
    print("BUILDING RELOCATION SUMMARY")
    print("="*50)
    print(f"Input parcels: {len(parcels)}")
    print(f"Output buildings: {len(output_gdf)}")
    print(f"\nParameters used:")
    print(f"  similar_size_ratio: {similar_size_ratio}")
    print(f"  min_building_sqm: {min_building_sqm}")
    print(f"\nBy relocation rule:")
    print(output_gdf['relocation_rule'].value_counts())
    
    res3_adjusted = output_gdf[output_gdf['final_occtype'] != output_gdf['original_occtype']]
    if len(res3_adjusted) > 0:
        print(f"\nRES3 occupancy adjustments: {len(res3_adjusted)}")
        print(res3_adjusted.groupby(['original_occtype', 'final_occtype']).size())
    
    print("="*50)
    
    return output_gdf

In [67]:
SIMILAR_SIZE_RATIO = 0.75  # If secondary > 75% of primary, consider similar size
MIN_BUILDING_SQM = nj_buildings.geometry.area.quantile(0.01)     # Filter tiny sheds/garages as 1% quantile
MIN_OVERLAP_RATIO = 0.075 # small buffer for reducing tiny overlaps (7.5 cm)

In [68]:
%%time
final_buildings = relocate_buildings_with_footprints(parcel_agg, nj_buildings,
                                                    similar_size_ratio=SIMILAR_SIZE_RATIO,
                                                    min_building_sqm=MIN_BUILDING_SQM,
                                                    min_overlap_ratio=MIN_OVERLAP_RATIO)


BUILDING RELOCATION SUMMARY
Input parcels: 2480307
Output buildings: 2480307

Parameters used:
  similar_size_ratio: 0.75
  min_building_sqm: 131.23363310189677

By relocation rule:
relocation_rule
parcel_centroid    2480307
Name: count, dtype: int64
CPU times: user 2min 23s, sys: 6.55 s, total: 2min 29s
Wall time: 2min 30s


In [69]:
final_buildings

,GIS_PIN,nsi_val_struct,nsi_val_cont,LAND_VAL,IMPRVT_VAL,NET_VALUE,final_occtype,original_occtype,parcels_spanned,final_stories,...,has_basement,found_ht_q05,found_ht_q25,found_ht_q50,found_ht_q75,found_ht_q95,nsi_count,fp_id,relocation_rule,geometry
0,0101_1.01_1.01,227441.560,113720.7800,70000.0,158500.0,228500.0,RES1,RES1,1,2.0,...,True,0.21,0.44,0.65,0.90,1.23,1,None,parcel_centroid,POINT (-74.505 39.449)
1,0101_1.01_1.02,241118.257,120559.1280,70000.0,144000.0,214000.0,RES1,RES1,1,2.0,...,True,0.20,0.43,0.64,0.88,1.23,1,None,parcel_centroid,POINT (-74.505 39.448)
2,0101_1.01_10,534721.042,267360.5210,70000.0,157600.0,227600.0,RES1,RES1,1,2.0,...,True,0.18,0.42,0.61,0.88,1.21,1,None,parcel_centroid,POINT (-74.508 39.448)
3,0101_1.01_11,370374.877,185187.4380,70000.0,134700.0,204700.0,RES1,RES1,1,2.0,...,True,0.19,0.45,0.65,0.89,1.21,1,None,parcel_centroid,POINT (-74.509 39.448)
4,0101_1.01_13,187762.983,93881.4918,70000.0,133400.0,203400.0,RES1,RES1,1,2.0,...,True,0.19,0.43,0.64,0.89,1.25,1,None,parcel_centroid,POINT (-74.507 39.446)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2480302,2123_9_2.03,259658.161,129829.0800,107400.0,235100.0,342500.0,RES1,RES1,1,2.0,...,True,0.18,0.42,0.62,0.87,1.23,1,None,parcel_centroid,POINT (-75.093 40.784)
2480303,2123_9_2.04,219644.988,109822.4940,107400.0,213100.0,320500.0,RES1,RES1,1,2.0,...,True,0.19,0.44,0.64,0.89,1.24,1,None,parcel_centroid,POINT (-75.093 40.784)
2480304,2123_9_2.05,154698.237,77349.1187,107500.0,126400.0,233900.0,RES1,RES1,1,1.0,...,False,0.57,1.24,1.78,2.41,3.33,1,None,parcel_centroid,POINT (-75.093 40.783)
2480305,2123_9_3,170419.914,85209.9573,100500.0,108200.0,208700.0,RES1,RES1,1,2.0,...,True,0.20,0.44,0.64,0.90,1.24,1,None,parcel_centroid,POINT (-75.092 40.783)


In [70]:
final_buildings.relocation_rule.value_counts()

relocation_rule
parcel_centroid    2480307
Name: count, dtype: int64

## Step 8 - Assign DDFs based on the final characteristics we derived
Finally, we can assign the DDF based on the HAZUS average DDFs Alexander calculated for the CONUS risk study. While the most rigorous approach would be to do this probabalistically and sample for all the different curves, this is unnecessary for the more crude assessment we are doing for Ida. A DDF was added for split level w basement and no basement and 1.5 story properties. For simplicity, since the 1.5 story DDFs were not available for basement/no basement and we cannot know the level of the 0.5 story (ie is it a basement or upper story), we just use the USACE Wilmington DDF with no foundation type specified as basement/no basement.

Indeed, we already skipped other probabalistic steps, and if desired we can simplify even further by taking the median of the monte carlo simulation for the foundation heights.

In [71]:
# read in the lookup table for the DDFs
ddf_lookup = pd.read_csv("/scratch/gpfs/GVILLARI/lt0663/risk_analysis/raritan_parcels/hazus_mapping_raritan_enh.csv")
ddf_lookup.head(5)

,Occupancy,occtype,Number of Stories,Basement,Source,Description,Structure Function ID,Content Function ID
0,AGR1,AGR1,"Low-, Mid-, or HighRise",Basement or No Basement,USACE Galveston,"Average of 3 Agriculture classes, structure",616,460
1,COM1,COM1,"Low-, Mid-, or HighRise",Basement or No Basement,USACE Galveston,"Average of 47 Retail classes, structure",217,90
2,COM10,COM10,"Low-, Mid-, or HighRise",Basement or No Basement,USACE Galveston,"Garage, structure",543,357
3,COM2,COM2,"Low-, Mid-, or HighRise",Basement or No Basement,USACE Galveston,"Average of 22 wholesale/warehouse classes, str...",341,195
4,COM3,COM3,"Low-, Mid-, or HighRise",Basement or No Basement,USACE Galveston,Average of 16 personal & repair service classe...,375,240


In [72]:
def get_structure_ddf_id(occtype, stories, has_basement, ddf_lookup):
    """
    Get Structure Function ID from DDF lookup table.
    """
    if pd.isna(occtype):
        return None
    
    # Filter to this occtype
    matches = ddf_lookup[ddf_lookup['occtype'] == occtype]
    
    if len(matches) == 0:
        return None
    
    # If only one match, return it (non-RES1 types)
    if len(matches) == 1:
        return matches['Structure Function ID'].iloc[0]
    
    # For RES1: need to match stories and basement
    if occtype == 'RES1':
        # Normalize stories to string for lookup
        if pd.isna(stories):
            stories_match = '1'
        elif stories in [-0.5, 0.5]:
            stories_match = '-0.5'
        elif stories < 1.5:
            stories_match = '1'
        elif stories == 1.5:
            stories_match = '1.5'
        elif stories < 3:
            stories_match = '2'
        else:
            stories_match = '3'
        
        matches = matches[matches['Number of Stories'] == stories_match]
    
    # Match basement
    if len(matches) > 1:
        if has_basement == True:
            bsmt_matches = matches[
                (matches['Basement'] == 'Basement') | 
                (matches['Basement'].str.startswith('Basement', na=False))
            ]
        else:
            bsmt_matches = matches[matches['Basement'].str.contains('No Basement', na=False)]
        
        if len(bsmt_matches) > 0:
            matches = bsmt_matches
    
    if len(matches) > 0:
        return matches['Structure Function ID'].iloc[0]
    
    return None

In [73]:
def assign_ddf_ids(df, ddf_lookup):
    """
    Assign Structure Function ID to each row.
    """
    df = df.copy()
    
    df['structure_ddf_id'] = df.apply(
        lambda r: get_structure_ddf_id(
            r['final_occtype'], 
            r['final_stories'], 
            r['has_basement'],
            ddf_lookup
        ),
        axis=1
    )
    
    print(f"DDF assigned: {df['structure_ddf_id'].notna().sum()} / {len(df)}")
    
    return df

In [74]:
%%time
final_buildings_ddfs = assign_ddf_ids(final_buildings, ddf_lookup)
final_buildings_ddfs.head(5)  # inspect this for what the final data looks like!

DDF assigned: 2480307 / 2480307
CPU times: user 19min 56s, sys: 26.5 s, total: 20min 23s
Wall time: 20min 6s


,GIS_PIN,nsi_val_struct,nsi_val_cont,LAND_VAL,IMPRVT_VAL,NET_VALUE,final_occtype,original_occtype,parcels_spanned,final_stories,...,found_ht_q05,found_ht_q25,found_ht_q50,found_ht_q75,found_ht_q95,nsi_count,fp_id,relocation_rule,geometry,structure_ddf_id
0,0101_1.01_1.01,227441.560,113720.7800,70000.0,158500.0,228500.0,RES1,RES1,1,2.0,...,0.21,0.44,0.65,0.90,1.23,1,None,parcel_centroid,POINT (-74.505 39.449),108
1,0101_1.01_1.02,241118.257,120559.1280,70000.0,144000.0,214000.0,RES1,RES1,1,2.0,...,0.20,0.43,0.64,0.88,1.23,1,None,parcel_centroid,POINT (-74.505 39.448),108
2,0101_1.01_10,534721.042,267360.5210,70000.0,157600.0,227600.0,RES1,RES1,1,2.0,...,0.18,0.42,0.61,0.88,1.21,1,None,parcel_centroid,POINT (-74.508 39.448),108
3,0101_1.01_11,370374.877,185187.4380,70000.0,134700.0,204700.0,RES1,RES1,1,2.0,...,0.19,0.45,0.65,0.89,1.21,1,None,parcel_centroid,POINT (-74.509 39.448),108
4,0101_1.01_13,187762.983,93881.4918,70000.0,133400.0,203400.0,RES1,RES1,1,2.0,...,0.19,0.43,0.64,0.89,1.25,1,None,parcel_centroid,POINT (-74.507 39.446),108


### Breakdown of the final dataset:
* `GIS_PIN`: a list of all the GIS_PIN values from the parcel data that are incorporated into this structure. Many GIS_PINs indicates a footprint crosses multiple tax parcels. While this might be an error (especially if it is RES1 like our first entry - may be a RES3 apartment complex or RES2 manufactured home park in reality, or just a footprint that marginally encroaches on a neighbouring footprint; see data quality discussion) it could just as easily be correct. A good place to confirm this manually is in Mercer County near Princeton, where there a lot of very large properties spanning multiple parcels.
* `LAND_VAL`: the value of the LAND in the parcel without any buildings. This is probably not relevant for our risk assessment.
* `IMPRVT_VAL`: the value of the IMPROVEMENTS to the land, which depreciates. While I cannot find the official wording to confirm this, it appears to me that given this is the appraised value input by the tax assessor, this can be treated as analagous to `val_struct` in NSI. Indeed, when comparing losses between the two datasets, the content losses should be left out and only calculated for `val_struct` (unless we want to estimate content value using the HAZUS algorithms).
* `NET_VALUE`: the land value plus the improvement value.
* `final_occtype`: the final HAZUS occtype parsed from the merging of NSI, parcel and footprint data.
* `original_occtype`: basically a data quality flag for aggregated RES3 properties. This could probably be more useful and reference the NSI or raw parcel data but is not relevant to final risk assessment so not a priority...
* `parcels_spanned`: tells you the number of parcels that have been aggregated into the footprint centroid.
* `final_stores`: tells you the final number of stories parsed from the merging of NSI, parcel and footprint data.
* `final_foundation`: tells you the final foundation type parsed from the merging of NSI, parcel and footprint data (NOTE: maybe i should have done the FFE MC here?? Oh well, can move if we think is necessary as is easy to run again)
* `has_basement`: is the final foundation type a basement?
* `found_ht_qX`: the foundation height quantiles from the MC simulation. Note that in the final "minimal" data where much of the metadata encoded in this dataframe is dropped, only the median foundation height is kept. You might want to change to something more or less conservative, or just use all of them so there are error bars.
* `nsi_count`: how many NSI points were aggregated into the final structure?
* `fp_id`: footprint ID (used where a footprint was used but in the few cases where we just have NSI and parcel will be None)
* `relocation_rule`: what rule was used if the point representing the structure was relocated?
* `geometry`: the geometry of the points representing the final structures.

### DATA AND PATHS
1. `/scratch/gpfs/GVILLARI/lt0663/risk_analysis/nj_parcels/nj_inventory` is the base path for all of these
2. `/scratch/gpfs/GVILLARI/lt0663/risk_analysis/nj_parcels/nj_inventory/nj_inventory_full_no_obm.gpkg` was calculated ONLY from NSI and Parcel data
3. `/scratch/gpfs/GVILLARI/lt0663/risk_analysis/nj_parcels/nj_inventory/nj_inventory_full_w_obm_adj.gpkg` includes the OBM footprint adjustment. I would like to see the difference between these for final estimates.
4. `/scratch/gpfs/GVILLARI/lt0663/risk_analysis/raritan_parcels/raritan_inventory/hazus_mapping_raritan_enh.csv` was the lookup table for DDF assignment. This should work directly with the DDF csv Alexander made that was used for NSI.
5. The `full` data includes all the quality flags etc. The `minimal` data includes only basic information.

In [75]:
final_buildings_ddfs.structure_ddf_id.value_counts()

structure_ddf_id
108    666306
133    562738
205    313801
107    241307
129    229042
204    137403
176     98994
110     36824
631     36579
217     36018
431     22206
109     21677
616     17230
545     15402
375      9419
111      8497
624      8398
189      4760
112      2910
643      2901
475      2222
473      1367
209       932
543       921
493       870
559       442
215       342
474       263
575       242
592       134
214        53
640        42
586        38
652        15
341        11
532         1
Name: count, dtype: int64

The top 5 most common DDFs are:
1. two stories with basement
2. one story with with basement
3. two stories with no basement
4. one story with no basement
5. 1.5 stories w or without basement

Based on the prevalence of the 1.5 story property, which can only have come from the parcel data, we probably ought to look into what this DDF really entails (e.g. is the DDF actually valid for basement/no basement?)

In [76]:
OUTPATH = f"/scratch/gpfs/GVILLARI/lt0663/risk_analysis/nj_parcels/nj_inventory"

MINIMAL_COLS = [
    "GIS_PIN", 'LAND_VAL', 'IMPRVT_VAL', 'found_ht_q50',
    'final_occtype', 'final_stories', 'final_foundation', 'structure_ddf_id',
    'geometry'
]

In [77]:
if final_buildings_ddfs.crs is None:
    final_buildings_ddfs = final_buildings_ddfs.set_crs("EPSG:6347") 

In [78]:
# write the full dataset including the quality control flags
final_buildings_ddfs.to_file(f"{OUTPATH}/nj_inventory_full_w_obm_adj.gpkg")
# write a minimal version of the dataset without flags etc
#final_buildings_ddfs[MINIMAL_COLS].to_file(f"{OUTPATH}/nj_inventory_minimal_w_obm_adj.gpkg")

Final considerations/questions we should ask ourselves (please add if I did things that didn't make sense):
* is the improvement value really comparable to the replacement cost calculated in NSI? How do we phrase this?
* this data is not homogenous in time and may not fully represent the built environment in 2021. Rutgers holds an archive of MOD-IV data joined with parcels, but as far as I can see NJ does not. I requested the archive from Rutgers as it was too large to download myself, but they have yet to respond. In lieu of this, do we want to adjust these parcels for inflation? They should represent the 2024 tax year. Using [CPI inflation calculator](https://www.bls.gov/data/inflation_calculator.htm) 1 USD in September 2021 is worth 1.06 USD in 2024 USD. There might also be a different way to adjust for inflation in property values (which in NJ I assume has been insane post-pandemic) but I am not really sure how that factors in here?
* Is the uncertainty here acceptable? How does this compare to the NSI structure values alone? How will we acknowledge the uncertainty given all the assumptions we had to make? Down the line, what can State of NJ provide that we might be able to use as a reference to reduce this uncertainty.
* How important is content value in the loss calculation? If we think content value is very important then should we calculate it using the HAZUS formulas (uses occtype and square footage with a scaling factor and depreciation). I think Pollack et al., 2025 study that is a more comprehensive version of this analysis (due to the better parcel data they had available) only uses the structure value, so I think that should be acceptable here too: see for more on that https://github.com/abpoll/nsi_fit/blob/main/notebooks/prepare_data.ipynb


## Explore the data quality
When I have time I will add a detailed exploration of the data quality based on things I find in QGIS. For now though, I have a few arbitrary examples of common cases I have found that are frustrating and I don't have a better way to deal with than accepting some level of uncertainty. I think it would be good to at the very least add some error bars on the risk number by assuming 30% standard deviation either side of the DDFs and using the different FFEs, but ultimately this is probably dependent on space and time.